<a href="https://colab.research.google.com/github/gurveervirk/CodeSage/blob/main/using_inspect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Dependencies

In [ ]:
!pip install -q importlib typing llama-index-graph-stores-neo4j importlib-metadata astroid llama-index-llms-gemini==0.4.5 llama-index-embeddings-gemini llama-index-embeddings-huggingface ratelimit aiolimiter flake8 llama-index-agent-lats llama-index-tools-duckduckgo

# COMStore (Code Object Model Store)

In [ ]:
"""
This module is an integration of COM logic with the Neo4jPropertyGraphStore class.
"""
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core.bridge.pydantic import PrivateAttr
from llama_index.core.graph_stores.types import (
    EntityNode, Relation
)
from llama_index.core import Settings
from typing import Any, Dict, List, Optional, Tuple
# from reflex import Entity, Relationship
import re
import logging
import neo4j

logging.basicConfig(level=logging.INFO)
BASE_ENTITY_LABEL = "__Entity__"
BASE_NODE_LABEL = "__Node__"

class Entity:
    """
    Module-agnostic entity class.
    """
    def __init__(self, name: str, label: str, properties: Dict):
        self.name = name
        self.label = label
        self.properties = properties

    def __repr__(self):
        return f"{self.label} ({self.name})"

class Relationship:
    """
    Module-agnostic relation class.
    """
    def __init__(self, from_entity: Entity, to_entity: Entity, label: str, properties: Dict):
        self.from_entity = from_entity
        self.to_entity = to_entity
        self.label = label
        self.properties = properties

    def __repr__(self):
        return f"{self.from_entity} --{self.label}--> {self.to_entity}"


def check_if_not_null(props: List[str], values: List[Any]) -> None:
    """Check if variable is not null and raise error accordingly."""
    for prop, value in zip(props, values):
        if not value:
            raise ValueError(f"Parameter `{prop}` must not be None or empty string")

# Template for generating code for the entity with all of it's relevant entities
DEFAULT_CODE_GENERATION_PROMPT_TEMPLATE = """
    You are a code generation expert. You are to generate end-to-end code, with all the dependencies and requirements, for the following MAIN_ENTITY, using the following information:

    MAIN_ENTITY:
    {main_entity_name}

    RELEVANT_ENTITIES:
    {relevant_entities_information}

    Please keep in mind the following instructions, if any:
    {other_instructions}

    Your code should be complete and ready to run.
    The code should be specifically and solely for the MAIN_ENTITY ONLY, NOTHING ELSE.
    Use the shortest_call_path's to import and use stuff, wherever necessary.
    This is a completely new and unique code being shown to you, used and generated. Do not worry about recitation.

    Code:
"""

# Template for getting the full paths of dependencies given an entity and it's MODULE code
DEFAULT_DEPENDENCY_PATHS_PROMPT_TEMPLATE = """
    You are a code analysis expert. You are to find all the full paths of parameter types as well as return types for using the following MAIN_ENTITY, given the following information, rules and examples ONLY:

    MAIN_ENTITY name:
    {main_entity_name}

    PARENT_MODULE information:
    {parent_module_information}

    **Rules:**

    These rules are designed to precisely filter the dependencies you need to report. Follow them in the order presented, with higher-numbered rules having precedence over lower-numbered rules.

    **1. Entity Type Determination:**

    -   Determine whether the MAIN_ENTITY is a **Function/Method** or a **Class**.

    **2. Handling Functions/Methods:**

    -   If the MAIN_ENTITY is a **Function/Method:** (**THESE INCLUDE EVERYTHING STARTING WITH A `def ` in python, from methods, functions to properties, etc.**)
        -   **Identify:**
            -   Parameter types from the function/method signature.
            -   Return types from the function/method signature or docstring (if present in the format `:rtype:`).
        -   **Include:** Only the identified parameter and return types.
        -   **Exclude:**
            -   Any types related to exceptions or errors.
            -   Types of any intermediate variables.

    **3. Handling Classes:**

    -   If the MAIN_ENTITY is a **Class:**
        -   **Analyze:** All methods within the class, excluding the `__init__` method.
        -   **For each method (**THESE INCLUDE EVERYTHING STARTING WITH A `def ` in python, from methods, functions to properties, etc.**):**
            -   **Identify:**
                -   Parameter types from the method signature.
                -   Return types from the method signature or docstring (if present in the format `:rtype:`).
            -   **Include:** Only the identified parameter and return types from each method.
            -   **Exclude:**
                -   Any types related to exceptions or errors.
                -   Types of any intermediate variables.
        -   **Special Handling of `__init__`:**
            -   **Identify:** Parameter types from the `__init__` method's signature.
            -   **Include:** Only the identified parameter types.
            -   **Exclude:** Everything else from `__init__`, including return types (which is implicitly `None`).
        -   **Class/Instance Attribute Exclusion:**
            -   **Always Exclude:** Types used to define or initialize class or instance attributes, variables, or properties. These are often found in the `__init__` method or defined directly within the class using `self.`.

    **4. Dependency Filtering:**

    -   After applying rules 1-3, filter the identified types as follows:
        -   **Exclude:**
            -   **Built-in Types:** Types from the `builtins` module (e.g., `str`, `int`, `list`, `None`).
            -   **Typing Module Types:** Types from the `typing` module (e.g., `List`, `Dict`, `Optional`).
            -   **Implicit Dependencies:** Types that are not explicitly used as parameter or return types.
            -   **Reiterate Class/Instance Attribute Exclusion:** Exclude any types that were used to define or initialize class or instance attributes, variables, or properties, even if they appear in a method's parameter or return type.

    Example 1:
    MAIN_ENTITY name:
    alpha.beta.gamma

    PARENT_MODULE information:
    Name: beta
    Full Path: alpha.beta
    Imports: ['charlie.delta']
    Source: ```
    from charlie import delta

    def something() -> None:
        pass

    def gamma(in: delta) -> None:
        something()
        pass
    ```

    Paths:
    charlie.delta

    Example 2:
    MAIN_ENTITY name:
    alpha.beta.gamma

    PARENT_MODULE information:
    Name: beta
    Full Path: alpha.beta
    Imports: ['charlie.delta']
    Source: ```
    from charlie import delta

    def gamma() -> None:
        pass
    ```

    Paths:
    NONE

    Example 3:
    MAIN_ENTITY name:
    alpha.beta.gamma

    PARENT_MODULE information:
    Name: beta
    Full Path: alpha.beta
    Imports: ['charlie.delta']
    Source: ```
    from charlie import delta

    def gamma():
        ```
        A function that uses something.

        :rtype: delta
        ```
        return something()

    Paths:
    charlie.delta

    Example 4:
    MAIN_ENTITY name:
    alpha.beta.gamma

    PARENT_MODULE information:
    Name: beta
    Full Path: alpha.beta
    Imports: ['charlie.delta']
    Source: ```
    from charlie import delta

    def gamma():
        ```
        A function that does something.

        :rtype: something
        ```
        return delta()

    Paths:
    something

    Example 5:
    MAIN_ENTITY name:
    alpha.beta.gamma

    PARENT_MODULE information:
    Name: beta
    Full Path: alpha.beta
    Imports: ['charlie.delta', 'charlie.epsilon', 'charlie.zeta', 'charlie.eta', 'charlie.theta']
    Source: ```
    from charlie import delta, epsilon, zeta, eta, theta

    class gamma:
        ```
        A class that does something.
        ```
        def __init__(self, something: delta):
            self.something = something
            self.something_else = epsilon()

        def method_1(self, something_else: zeta) -> eta:
            temp = theta(something_else)

            return eta(temp)
    ```

    Paths:
    charlie.delta
    charlie.zeta
    charlie.eta

    Please think, reason and analyze the above properly.
    Your response should be formatted as follows:
    <thinking>
    (all thinking, reasoning, analysis to be written here IN DETAIL)
    </thinking>

    Paths:
    (dependencies listed here, one per line)
"""

class COMStore(Neo4jPropertyGraphStore):
    """
    COM stands for Code Object Model.
    This class is a subclass of Neo4jPropertyGraphStore and is used to store and retrieve code entities and their relationships in a Neo4j database.
    It provides ways to create, update, and query entities and relationships in the database. The database stores the Python Modular Tree structure in the form of nodes and relationships.

    Schema:
    - Nodes: MODULE, CLASS, METHOD
    - Relationships: HAS_CLASS, HAS_METHOD, INHERITS, IMPORTS
    """
    stores_text: bool = True
    flat_metadata: bool = True
    distance_strategy: str
    index_name: str
    keyword_index_name: str
    hybrid_search: bool
    node_label: str
    embedding_node_property: str
    text_node_property: str
    retrieval_query: str
    _support_metadata_filter: bool = PrivateAttr()
    _is_enterprise: bool = PrivateAttr()
    custom_fts_indices: Any = []

    def __init__(
        self,
        username: str,
        password: str,
        url: str,
        database: Optional[str] = "neo4j",
        index_name: str = "entity",
        keyword_index_name: str = "keyword",
        node_label: str = "__Node__",
        embedding_node_property: str = "embedding",
        text_node_property: str = "source",
        retrieval_query: str = "",
        refresh_schema: bool = True,
        sanitize_query_output: bool = True,
        enhanced_schema: bool = False,
        **neo4j_kwargs: Any,
    ) -> None:
        self.index_name = index_name
        self.keyword_index_name = keyword_index_name
        self.node_label = node_label
        self.embedding_node_property = embedding_node_property
        self.text_node_property = text_node_property
        self.retrieval_query = retrieval_query
        self.database = database
        super().__init__(
            username,
            password,
            url,
            database,
            refresh_schema,
            sanitize_query_output,
            enhanced_schema,
            **neo4j_kwargs,
        )
        check_if_not_null(
            [
                "index_name",
                "node_label",
                "embedding_node_property",
                "text_node_property",
            ],
            [index_name, node_label, embedding_node_property, text_node_property],
        )
        self.create_required_indices()

    def create_required_indices(self) -> None:
        """
        This method constructs Cypher queries and executes them to create:
        1. 5 full-text indices in Neo4j:
            - one for ROOT_MODULE entities on 'name' and 'source' properties.
            - one for MODULE entities on 'name' and 'source' properties.
            - one for FUNCTION entities on 'name' and 'source' properties.
            - one for CLASS entities on 'name' and 'source' properties.
            - one for METHOD entities on 'name' and 'source' properties.
        2. 5 vector indices:
            - one for ROOT_MODULE entities on the 'embedding' property.
            - one for FUNCTION entities on the 'embedding' property.
            - one for MODULE entities on the 'embedding' property.
            - one for CLASS entities on the 'embedding' property.
            - one for METHOD entities on the 'embedding' property.

        It uses the IF NOT EXISTS clause to avoid errors if the indices already exist.
        """

        # Create full-text indices
        root_module_index_query = """
        CREATE FULLTEXT INDEX root_module_index IF NOT EXISTS FOR (n:`ROOT_MODULE`) ON EACH [n.`name`]
        """
        self.database_query(root_module_index_query)

        function_index_query = """
        CREATE FULLTEXT INDEX function_index IF NOT EXISTS FOR (n:`FUNCTION`) ON EACH [n.`name`]
        """
        self.database_query(function_index_query)

        module_index_query = """
        CREATE FULLTEXT INDEX module_index IF NOT EXISTS FOR (n:`MODULE`) ON EACH [n.`name`]
        """
        self.database_query(module_index_query)

        class_index_query = """
        CREATE FULLTEXT INDEX class_index IF NOT EXISTS FOR (n:`CLASS`) ON EACH [n.`name`]
        """
        self.database_query(class_index_query)

        method_index_query = """
        CREATE FULLTEXT INDEX method_index IF NOT EXISTS FOR (n:`METHOD`) ON EACH [n.`name`]
        """
        self.database_query(method_index_query)

        # Create vector indices
        root_module_vector_index_query = """
        CREATE VECTOR INDEX root_module_vector_index IF NOT EXISTS
        FOR (m:`ROOT_MODULE`)
        ON (m.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(root_module_vector_index_query)

        function_vector_index_query = """
        CREATE VECTOR INDEX function_vector_index IF NOT EXISTS
        FOR (f:`FUNCTION`)
        ON (f.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(function_vector_index_query)

        module_vector_index_query = """
        CREATE VECTOR INDEX module_vector_index IF NOT EXISTS
        FOR (m:`MODULE`)
        ON (m.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(module_vector_index_query)

        class_vector_index_query = """
        CREATE VECTOR INDEX class_vector_index IF NOT EXISTS
        FOR (c:`CLASS`)
        ON (c.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(class_vector_index_query)

        method_vector_index_query = """
        CREATE VECTOR INDEX method_vector_index IF NOT EXISTS
        FOR (m:`METHOD`)
        ON (m.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(method_vector_index_query)

        general_fts_query = """
        CREATE FULLTEXT INDEX general_fts_index IF NOT EXISTS FOR (n:__Entity__) ON EACH [n.name, n.source]
        """
        self.database_query(general_fts_query)

    def retrieve_relevant_root_modules(self, query: str, top_k: int = 10) -> str:
        """
        Retrieve relevant root modules based on the query, using similarity search.

        Args:
            query (str): The query string.
            top_k (int): The number of results to return. Defaults to None.

        Returns:
            str: Relevant root modules based on the query formatted as a string.

        Example Output:
        ```
        Relevant Root Modules:

        1. module1: summary1
        2. module2: summary2
        ...
        ```
        """
        results = self.query_vector_index("root_module_vector_index", query, top_k=top_k)

        # Format the results as a string
        formatted_results = "Relevant Root Modules:\n\n" + "\n".join(
            [f"{i+1}. **{result['name']}**: {result['summary']}" for i, result in enumerate(results)]
        )

        return formatted_results

    def retrieve_relevant_children_in_subtree(self, query: str, parent_name: str) -> str:
        """
        Retrieve relevant IMMEDIATE children in the subtree of the given parent based on the query, using similarity search.

        Args:
            query (str): The query string.
            parent_name (str): The name of the parent entity.

        Returns:
            str: Relevant children in the subtree of the parent based on the query formatted as a string.

        Example Output:
        ```
        Relevant Children in Subtree:

        1. child1: summary1
        2. child2: summary2
        ...
        ```
        """
        # Embed the query
        query_vector = Settings.embed_model.get_query_embedding(query)

        # Use a cypher query to return all IMMEDIATE children of the parent entity, sorted by similarity to the query
        sort_and_return_children_query = """
            MATCH (parent {name: $parent_name})-[:HAS_SUBMODULE|HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES]->(child)
            WHERE child.embedding IS NOT NULL
            WITH parent, child,
            vector.similarity.cosine(parent.embedding, $query_vector) AS similarity
            ORDER BY similarity DESC
            RETURN child.name AS name, similarity AS score, labels(child) AS labels, child.summary AS summary
        """

        query_params = {
            "parent_name": parent_name,
            "query_vector": query_vector,
        }

        try:
            records = self.database_query(sort_and_return_children_query, query_params)

            # Filter out base labels and retain only the relevant labels
            for record in records:
                record["labels"] = [label for label in record["labels"] if label not in [BASE_ENTITY_LABEL, BASE_NODE_LABEL]]

            # Format the results as a string
            formatted_results = "Relevant Children in Subtree:\n\n" + "\n".join(
                [f"{i+1}. **{result['name']}**: {result['summary']}" for i, result in enumerate(records)]
            )

            return formatted_results

        except Exception as e:
            logging.error(f"Error retrieving relevant children in subtree: {e}")
            return []

    def retrieve_full_name(self, entity_name: str) -> str:
        """
        Retrieve the full name of the entity usin general full-text search.

        Args:
            entity_name (str): The name of the entity.

        Returns:
            str: The full name of the entity.
        """
        return self.query_fulltext_index("general_fts_index", entity_name)[0]["name"]

    def query_vector_index(
        self, index_name: str, query: str | List[float], top_k: int = 10,
        **kwargs: Any
    ) -> List[Dict]:
        """
        Query a vector index.

        Args:
            index_name (str): The name of the vector index, restricted to `method_vector_index`, `class_vector_index` and `module_vector_index`.
            query (str or List[float]): The query or query vector embedding.
            top_k (int): The number of results to return. Defaults to 10.

        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        query_vector = None

        if isinstance(query, str):
            query_vector = Settings.embed_model.get_query_embedding(query)
        else:
            query_vector = query

        return_fields = ["name", "summary", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        CALL db.index.vector.queryNodes($index_name, $top_k, $query_vector)
        YIELD node AS n, score
        RETURN {return_fields_str}, score
        """
        query_params = {
            "index_name": index_name,
            "top_k": top_k,
            "query_vector": query_vector,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            logging.error(f"Error querying vector index '{index_name}': {e}")
            return []

    def query_fulltext_index(
        self, index_name: str, query: str,
        **kwargs: Any
    ) -> List[Dict]:
        """
        Query a full-text index.

        Args:
            index_name (str): The name of the full-text index, restricted to `module_index`, `class_index` and `method_index`.
            query (str): The query string.
        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        return_fields = ["name", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        CALL db.index.fulltext.queryNodes($index_name, $query)
        YIELD node AS n, score
        RETURN {return_fields_str}, score
        """
        query_params = {
            "index_name": index_name,
            "query": query,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            logging.error(f"Error querying full-text index '{index_name}': {e}")
            return []

    def similarity_search_without_index(
        self,
        labels: List[str],
        query: str | List[float],
        top_k: int = 10,
        **kwargs: Any,
    ) -> List[Dict]:
        """
        Perform a similarity search without using an index.

        Args:
            labels (List[str]): A list of labels to query (e.g., ["MODULE", "CLASS", "METHOD"]).
            query (str or List[float]): The query or query vector embedding.
            top_k (int): The number of results to return. Defaults to 10.

        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        query_vector = None
        if isinstance(query, str):
            query_vector = Settings.embed_model.get_query_embedding(query)
        else:
            query_vector = query
        return_fields = ["name", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        MATCH (n)
        WHERE any(label in labels(n) WHERE label IN $labels)
        AND n.{self.embedding_node_property} IS NOT NULL
        WITH n,
        vector.similarity.cosine(n.{self.embedding_node_property}, $query_vector) AS similarity
        ORDER BY similarity DESC
        LIMIT $top_k
        RETURN {return_fields_str}, labels(n) AS all_labels, similarity AS score
        """

        query_params = {
            "labels": labels,
            "query_vector": query_vector,
            "top_k": top_k,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            logging.error(f"Error performing similarity search: {e}")
            return []

    def database_query(
        self,
        query: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> Any:
        """
        Execute a cypher query in the Neo4j database.

        Args:
            query (str): The query string.
            params (Optional[Dict[str, Any]]): A dictionary of parameters to pass to the query.
        """
        params = params or {}
        try:
            data, _, _ = self._driver.execute_query(
                query, database_=self.database, parameters_=params
            )
            return [r.data() for r in data]
        except neo4j.exceptions.Neo4jError as e:
            if not (
                (
                    (  # isCallInTransactionError
                        e.code == "Neo.DatabaseError.Statement.ExecutionFailed"
                        or e.code
                        == "Neo.DatabaseError.Transaction.TransactionStartFailed"
                    )
                    and "in an implicit transaction" in e.message
                )
                or (  # isPeriodicCommitError
                    e.code == "Neo.ClientError.Statement.SemanticError"
                    and (
                        "in an open transaction is not possible" in e.message
                        or "tried to execute in an explicit transaction" in e.message
                    )
                )
            ):
                raise
        # Fallback to allow implicit transactions
        with self._driver.session(database=self.database) as session:
            data = session.run(neo4j.Query(text=query), params)
            return [r.data() for r in data]

    def upsert_to_neo4j(
            self,
            module_name: str,
            entities: List[Entity],
            relations: List[Relationship],
            other_relations: Dict[str, List[Tuple]],
            embed_nodes: Optional[bool] = False
        ) -> None:
        """
        Upsert entities and relationships into Neo4j.

        Args:
            module_name (str): The name of the module.
            entities (List[Entity]): List of entity nodes to upsert.
            relations (List[Relationship]): List of relationships to upsert.
            other_relations (Dict[str, List[Tuple]]): Dictionary containing other relations.
        """
        # Convert Entity to EntityNode and Relationship to Relation
        entity_nodes = []
        entities_to_entity_nodes = {}

        for entity in entities:
            entity_node = EntityNode(
                label=entity.label,
                name=entity.name,
                properties=entity.properties
            )
            entity_nodes.append(entity_node)
            entities_to_entity_nodes[entity.name] = entity_node

        relations = [
            Relation(
                label=r.label,
                source_id=entities_to_entity_nodes[r.from_entity.name].id,
                target_id=entities_to_entity_nodes[r.to_entity.name].id,
                properties=r.properties
            ) for r in relations
        ]

        self.upsert_nodes(entity_nodes)
        self.upsert_relations(relations)

        # Embed the nodes if specified
        if embed_nodes:
            self._embed_nodes()

        # Add labels recursively
        self._handle_inherited_methods(module_name)

        # Add INHERITS and IMPORTS relations as properties to the respective nodes
        self._add_other_relations_as_properties(other_relations)

        # Process INHERITS and IMPORTS relations
        self._process_other_relations()

        # Set the shortest call paths for all nodes in the module
        self._set_shortest_call_paths(module_name)

        # Add dunder labels to entities starting with '_'
        self._add_dunder_labels(module_name)

    def _add_dunder_labels(self, module_name: str):
        """
        Add dunder labels to entities starting with '_' in the specified module's subtree.

        Args:
            module_name (str): The name of the module.
        """
        # Create cypher query to add dunder labels to entities starting with '_'
        add_dunder_labels_query = (
            """
            MATCH (root:ROOT_MODULE {name: $module_name})
            MATCH (root)-[:HAS_SUBMODULE|HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES*]->(children)
            WHERE children.name STARTS WITH '_'
            CALL apoc.create.addLabels(children, ['DUNDER'])
            YIELD node
            RETURN count(node) AS numNodesLabeled
            """
        )

        self.structured_query(add_dunder_labels_query, {"module_name": module_name})

    def _add_other_relations_as_properties(self, other_relations: Dict[str, List[Tuple]]):
        """
        Add INHERITS and IMPORTS relations as properties to the respective nodes.

        Args:
            other_relations (Dict[str, List[Tuple]]): Dictionary containing INHERITS and IMPORTS relations.
        """
        inherits_relations = other_relations.get("INHERITS", [])
        imports_relations = other_relations.get("IMPORTS", [])
        exposes_relations = other_relations.get("EXPOSES", [])
        add_inherits_query = (
            """
            UNWIND $inherits_relations AS rel
            MATCH (c:CLASS {name: rel[0]})
            SET c.INHERITS = apoc.coll.toSet(coalesce(c.INHERITS, []) + rel[1])
            """
        )
        add_imports_query = (
            """
            UNWIND $imports_relations AS rel
            MATCH (e {name: rel[0]})
            SET e.IMPORTS = apoc.coll.toSet(coalesce(e.IMPORTS, []) + rel[1])
            """
        )
        add_exposes_query = (
            """
            UNWIND $exposes_relations AS rel
            MATCH (m {name: rel[0]})
            SET m.EXPOSES = apoc.coll.toSet(coalesce(m.EXPOSES, []) + rel[1])
            """
        )
        self.structured_query(add_inherits_query, {"inherits_relations": inherits_relations})
        self.structured_query(add_imports_query, {"imports_relations": imports_relations})
        self.structured_query(add_exposes_query, {"exposes_relations": exposes_relations})

    def _process_other_relations(self):
        """
        Execute Cypher queries to create relations on existing nodes with INHERITS, IMPORTS and EXPOSES properties.
        Clear names from the INHERITS, IMPORTS and EXPOSES property arrays of the nodes if the relations could be created successfully.
        """
        process_inherits_query = (
            """
            MATCH (c:CLASS)
            WHERE c.INHERITS IS NOT NULL
            UNWIND c.INHERITS AS parent
            MATCH (p:CLASS {name: parent})
            MERGE (c)-[:INHERITS]->(p)
            WITH c, COLLECT(parent) AS parentNames
            SET c.INHERITS = [name IN c.INHERITS WHERE NOT name IN parentNames]
            """
        )
        process_imports_query = (
            """
            MATCH (e1)
            WHERE e1.IMPORTS IS NOT NULL
            UNWIND e1.IMPORTS AS importee
            MATCH (e2 {name: importee})
            MERGE (e1)-[:IMPORTS]->(e2)
            WITH e1, COLLECT(importee) AS importeeNames
            SET e1.IMPORTS = [name IN e1.IMPORTS WHERE NOT name IN importeeNames]
            """
        )
        process_exposes_query = (
            """
            MATCH (m:MODULE|ROOT_MODULE)
            WHERE m.EXPOSES IS NOT NULL
            UNWIND m.EXPOSES AS exposed
            MATCH (e {name: exposed})
            MERGE (m)-[:EXPOSES]->(e)
            WITH m, COLLECT(exposed) AS exposedNames
            SET m.EXPOSES = [name IN m.EXPOSES WHERE NOT name IN exposedNames]
            """
        )
        cleanup_empty_INHERITS_query = (
            """
            MATCH (n)
            WHERE n.INHERITS IS NOT NULL AND size(n.INHERITS) = 0
            REMOVE n.INHERITS
            """
        )
        cleanup_empty_IMPORTS_query = (
            """
            MATCH (n)
            WHERE n.IMPORTS IS NOT NULL AND size(n.IMPORTS) = 0
            REMOVE n.IMPORTS
            """
        )
        cleanup_empty_EXPOSES_query = (
            """
            MATCH (n)
            WHERE n.EXPOSES IS NOT NULL AND size(n.EXPOSES) = 0
            REMOVE n.EXPOSES
            """
        )
        add_names_to_EXPOSES_rels_query = (
            """
            MATCH (n)-[r:EXPOSES]-(m)
            WHERE r.name IS NULL
            SET r.name = n.name + "." + m.__name__
            RETURN count(r)
            """
        )

        self.structured_query(process_inherits_query)
        self.structured_query(process_imports_query)
        self.structured_query(process_exposes_query)
        self.structured_query(cleanup_empty_INHERITS_query)
        self.structured_query(cleanup_empty_IMPORTS_query)
        self.structured_query(cleanup_empty_EXPOSES_query)
        self.structured_query(add_names_to_EXPOSES_rels_query)

    def _set_shortest_call_paths(self, module_name: str):
        """
        Set the shortest call paths for all nodes in the module.

        Args:
            module_name (str): The name of the module.
        """
        # Set the shortest call paths for all nodes in the module tree (recursively)
        set_shortest_call_paths_query = (
            """
            MATCH (root:ROOT_MODULE {name: 'emoji'})
            MATCH (end:MODULE|FUNCTION|CLASS|METHOD)<-[:HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES*]-(root)
            WITH root, collect(end) AS endNodes
            CALL apoc.path.spanningTree(root, {
            relationshipFilter: 'HAS_SUBMODULE|HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES>',
            terminatorNodes: endNodes
            })
            YIELD path
            WITH nodes(path) AS nodesInPath
            WITH nodesInPath[-1] AS targetNode, nodesInPath[-2] as parentNode
            WHERE targetNode.shortest_call_path IS NULL
            SET targetNode.shortest_call_path =
            CASE
                WHEN size(parentNode.name + "." + targetNode.__name__) < size(targetNode.name)
                THEN parentNode.name + "." + targetNode.__name__
                ELSE targetNode.name
            END
            RETURN count(targetNode) AS numPathsSet
            """
        )
        numPathsSet = self.structured_query(set_shortest_call_paths_query, {"module_name": module_name})[0]["numPathsSet"]
        while numPathsSet > 0:
            numPathsSet = self.structured_query(set_shortest_call_paths_query, {"module_name": module_name})[0]["numPathsSet"]

    def _handle_inherited_methods(self, module_name: str):
        """
        Create HAS_METHOD relations for inherited methods for all classes in the specified module.

        Args:
            module_name (str): The name of the module.

        """
        handle_inheritance_query = (
            """
            MATCH (base:ROOT_MODULE {name: $module_name})-[r*0..]->(child:CLASS)
            WHERE NONE(rel IN r WHERE type(rel) = 'INHERITS')
            WITH DISTINCT child
            MATCH (child)-[:INHERITS*0..]->(parent:CLASS)
            WITH child, parent
            MATCH (parent)-[:HAS_METHOD]->(member)
            WHERE NOT EXISTS {
                MATCH (child)-[:HAS_METHOD]->(childMethod)
                WHERE childMethod.__name__ = member.__name__
            }
            WITH member, child
            MERGE (child)-[:HAS_METHOD]->(member)
            RETURN child.name AS ChildClass, COUNT(DISTINCT member) AS TotalParentMembersLabeled
            """
        )
        self.structured_query(handle_inheritance_query, {"module_name": module_name})

    def _remove_empty_values(self, input_dict):
        """
        Remove entries with empty values from the dictionary.

        Parameters:
        input_dict (dict): The dictionary from which empty values need to be removed.

        Returns:
        dict: A new dictionary with all empty values removed.
        """
        # Create a new dictionary excluding empty values
        return {key: value for key, value in input_dict.items() if value}

    def _embed_nodes(self):
        """
        Embed newly created nodes.
        """
        retrieve_new_nodes = (
            """
            MATCH (e)
            WHERE e.embedding IS NULL
            RETURN
            e.id AS name,
            labels(e)[0] AS type,
            properties(e) AS properties
            """
        )
        new_nodes = self.structured_query(retrieve_new_nodes)
        if not new_nodes:
            return

        logging.info(f"Found {len(new_nodes)} new nodes to embed.")

        entity_nodes, summaries = [], []
        # Create EntityNode objects for each node
        for node in new_nodes:
            entity_node = EntityNode(
                label=node["type"],
                name=node["name"],
                properties=self._remove_empty_values(node["properties"])
            )
            entity_nodes.append(entity_node)
            summaries.append(entity_node.properties.get("summary", ""))

        # Embed the nodes
        embeddings = Settings.embed_model.get_text_embedding_batch(
            summaries,
            show_progress=True
        )

        for i, node in enumerate(entity_nodes):
            node.properties[self.embedding_node_property] = embeddings[i]

        self.upsert_nodes(entity_nodes)

    def _get_parent_module_info(self, entity_name: str) -> List[str]:
        """
        Get the path from the given entity to the parent module.

        Args:
            entity_name (str): The name of the entity.

        Returns:
            List[str]: The path from the entity to the parent module.
        """
        get_parent_module_info_query = (
            """
            MATCH (end {name: $entity_name})<-[:HAS_CLASS|HAS_METHOD|HAS_FUNCTION*1..]-(parent:MODULE|ROOT_MODULE)
            WITH parent, parent.__name__ AS __name__, parent.name AS name, parent.source AS source, end.shortest_call_path AS shortest_call_path
            MATCH (parent)-[:IMPORTS]->(import)
            WITH __name__, name, source, shortest_call_path, collect(import.name) AS imports
            RETURN __name__, name, source, shortest_call_path, imports
            """
        )
        result = self.structured_query(get_parent_module_info_query, {"entity_name": entity_name})
        return result if result else []

    def _get_retrieved_entity_representation(self, retrieved_node: Dict) -> str:
        """
        Get the representation of the retrieved entity.

        Args:
            retrieved_node (Dict): The retrieved node.

        Returns:
            str: The representation of the retrieved entity.
        """
        entity_representation = f"""
        Name: {retrieved_node["__name__"]}
        Full Path: {retrieved_node["name"]}
        Imports: {retrieved_node["imports"]}
        Source: ```
        {retrieved_node["source"]}
        ```
        """
        return entity_representation

    def _parse_deps_response(self, response: str) -> List[str]:
        """
        Parse the dependencies response using regex.

        Response Format:
        <thinking>
        (all thinking, reasoning, analysis to be written here IN DETAIL)
        </thinking>

        Paths:
        (dependencies listed here, one per line)

        Args:
            response (str): The response string.

        Returns:
            List[str]: The parsed dependencies.
        """
        # Remove THINKING part of the response
        response = re.sub(r"<thinking>.*?</thinking>", "", response, flags=re.DOTALL)
        response = response.strip()

        # Extract the dependencies from the response
        dependencies = re.findall(r"Paths:\n(.*?)", response, flags=re.DOTALL)

        if dependencies:
            if "NONE" in dependencies[0]:
                logging.info("No dependencies found.")
                return []

            dependencies = dependencies[0].strip().split("\n")
            logging.info(f"Dependencies found: {dependencies}")

        return dependencies

    async def _get_dependencies(self, entity_name: str, parent_module_repr: str) -> List[str]:
        """
        Get the dependencies of the given entity.

        Args:
            entity_name (str): The name of the entity.
            parent_module_repr (str): The representation of the parent module.

        Returns:
            List[str]: The dependencies of the entity.
        """
        prompt = DEFAULT_DEPENDENCY_PATHS_PROMPT_TEMPLATE.format(
            main_entity_name=entity_name,
            parent_module_information=parent_module_repr
        )

        deps = await Settings.llm.acomplete(prompt)
        deps = deps.text.strip()

        deps = self._parse_deps_response(deps)
        return deps

    async def _generate_code_using_llm(self, main_entity_name: str, relevant_entities_repr: List[Tuple[str, str]], other_instructions: str) -> str:
        """
        Generate code using the LLM model.

        Args:
            main_entity_name (str): The name of the main entity.
            relevant_entities_repr (List[Tuple[str, str]]): The representations of the relevant entities.
            other_instructions (str): Other instructions to be included in the prompt.

        Returns:
            str: The generated code.
        """
        relevant_entities_information = ""
        for entity_repr in relevant_entities_repr:
            relevant_entities_information += f"""
            Shortest Call Path for {entity_repr[0]}: {entity_repr[2]}
            Module Code for {entity_repr[0]}:
            {entity_repr[1]}

            """

        prompt = DEFAULT_CODE_GENERATION_PROMPT_TEMPLATE.format(
            main_entity_name=main_entity_name,
            relevant_entities_information=relevant_entities_information,
            other_instructions=other_instructions
        )

        code = await Settings.llm.acomplete(prompt)
        code = code.text.strip()
        return code

    def _get_full_path_name(self, name: str, label: str) -> str:
        """
        Get the full path name of the entity.

        Args:
            name (str): The name of the entity.
            label (str): The label of the entity.

        Returns:
            str: The full path name of the entity.
        """
        retrieve_node_by_name_query = (
            """
            MATCH (n:METHOD|FUNCTION|CLASS {name: $name})
            WHERE $label = "" OR $label IN labels(n)
            RETURN n.name AS name
            """
        )

        # Execute the query to retrieve the node by name
        retrieved_node = self.structured_query(retrieve_node_by_name_query, {"name": name, "label": label})

        if not retrieved_node:
            # If no relevant nodes are found, try querying relationships with this name
            retrieve_relationship_by_name_query = (
                """
                MATCH (n)-[r:EXPOSES]->(m)
                WHERE r.name = $name
                RETURN m.name AS name
                """
            )

            # Execute the query to retrieve relevant relationships
            retrieved_node = self.structured_query(retrieve_relationship_by_name_query, {"name": name})

        if not retrieved_node:
            # If no relevant nodes or relationships are found, return an error message
            return "NONE"

        name = retrieved_node[0]["name"]
        return name

    async def generate_code(self, name: str, label: Optional[str] = "", other_instructions: Optional[str] =  None) -> str:
        """
        Generate code for a given entity name.

        Args:
            name (str): The name of the entity.
            label (Optional[str]): The label of the entity.
            other_instructions (Optional[str]): Other instructions to be included in the prompt.

        Returns:
            str: The generated code for the entity.
        """
        # Check if the entity exists in the database
        name = self._get_full_path_name(name, label)

        if name == "NONE":
            return "Entity not found in the database."

        parent_module = self._get_parent_module_info(name)
        parent_module_repr = self._get_retrieved_entity_representation(parent_module[0]) if parent_module else ""

        relevant_entities_repr = [(name, parent_module_repr, parent_module[0]["shortest_call_path"] if parent_module else "")]

        deps = await self._get_dependencies(name, parent_module_repr)
        unique_dependencies = set(deps)
        logging.info(f"Initial dependencies for entity {name}: {deps}")

        if other_instructions is None:
            # Get only first level dependencies
            for dep in deps:
                dep = self._get_full_path_name(dep, "")
                dep_parent_module = self._get_parent_module_info(dep)
                dep_parent_module_repr = self._get_retrieved_entity_representation(dep_parent_module[0]) if dep_parent_module else ""

                relevant_entities_repr.append((dep, dep_parent_module_repr, dep_parent_module[0]["shortest_call_path"] if dep_parent_module else ""))

            # Generate code using LLM
            return await self._generate_code_using_llm(name, relevant_entities_repr, "")

        while deps:
            next_deps = []

            for dep in deps:
                dep = self._get_full_path_name(dep, "")
                dep_parent_module = self._get_parent_module_info(dep)
                dep_parent_module_repr = self._get_retrieved_entity_representation(dep_parent_module[0]) if dep_parent_module else ""

                relevant_entities_repr.append((dep, dep_parent_module_repr, dep_parent_module[0]["shortest_call_path"] if dep_parent_module else ""))

                new_deps = await self._get_dependencies(dep, dep_parent_module_repr)
                for new_dep in new_deps:
                    new_dep = self._get_full_path_name(new_dep, "")
                    if new_dep not in unique_dependencies:
                        next_deps.append(new_dep)
                        unique_dependencies.add(new_dep)

            logging.info(f"Next level dependencies: {next_deps}")
            deps = next_deps

        return await self._generate_code_using_llm(name, relevant_entities_repr, other_instructions)

# Import required modules and initalize COM Store

In [ ]:
import nest_asyncio
import asyncio

nest_asyncio.apply()

In [ ]:
import sys
import logging

# logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))
logging.getLogger().setLevel(logging.INFO)

In [ ]:
# from llama_index.core import Settings
# from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
# from llama_index.core import PropertyGraphIndex
# from llama_index.core.graph_stores.types import EntityNode, Relation
# from llama_index.core.schema import MetadataMode
from google.colab import userdata
import google.generativeai as genai

genai.configure = getattr(genai.configure, "func", genai.configure)

# Initialize Neo4j connection
com_store =  COMStore(
    username="neo4j",
    password=userdata.get("NEO4J_PASSWORD_KG"),
    url=userdata.get("NEO4J_URI_KG"),
)

# Create and set GeminiWithRateLimits class as LLM

In [ ]:
from typing import Sequence, Optional, Dict, Any, Type
from llama_index.llms.gemini.utils import (
    chat_from_gemini_response,
    chat_message_to_gemini,
    completion_from_gemini_response,
)
from llama_index.llms.gemini import Gemini
from llama_index.core.base.llms.types import (
    ChatMessage,
    ChatResponse,
    CompletionResponse
)
from llama_index.core.bridge.pydantic import Field
from llama_index.core.constants import DEFAULT_NUM_OUTPUTS, DEFAULT_TEMPERATURE
from llama_index.core.callbacks import CallbackManager
from llama_index.core.instrumentation.events.llm import (
    LLMPredictEndEvent,
    LLMPredictStartEvent,
    LLMStructuredPredictInProgressEvent,
    LLMStructuredPredictEndEvent,
    LLMStructuredPredictStartEvent,
)
from llama_index.core.bridge.pydantic import (
    BaseModel,
    WithJsonSchema,
    Field,
    field_validator,
    model_validator,
    ConfigDict,
    ValidationError,
)
from llama_index.core.prompts import BasePromptTemplate, PromptTemplate
from aiolimiter import AsyncLimiter
import google.generativeai as genai
import llama_index.core.instrumentation as instrument

dispatcher = instrument.get_dispatcher(__name__)

GEMINI_MODELS = (
    "models/gemini-2.0-flash-001",
    "models/gemini-2.0-flash-exp",
    # Gemini 1.0 Pro Vision has been deprecated on July 12, 2024.
    # According to official recommendations, switch the default model to gemini-1.5-flash
    "models/gemini-1.5-flash",
    "models/gemini-1.5-flash-latest",
    "models/gemini-pro",
    "models/gemini-pro-latest",
    "models/gemini-1.5-pro",
    "models/gemini-1.5-pro-latest",
    "models/gemini-1.0-pro",
    # for some reason, google lists this without the models prefix
    "gemini-1.5-flash",
    "gemini-1.5-flash-latest",
    "gemini-1.0-pro",
)
from llama_index.core.utilities.gemini_utils import (
    merge_neighboring_same_role_messages,
)
from llama_index.llms.gemini import Gemini

class GeminiWithRateLimits(Gemini):
    rate_limit: int = Field(
        default=10,
        description="The number of allowed calls per period."
    )
    period: int = Field(
        default=60,
        description="The period in seconds for the rate limit."
    )
    limiter: AsyncLimiter = Field(
        default=None,
        exclude=True,
        description="The async rate limiter instance."
    )
    def __init__(
        self,
        api_key: Optional[str] = None,
        model: str = GEMINI_MODELS[0],
        temperature: float = DEFAULT_TEMPERATURE,
        max_tokens: Optional[int] = None,
        generation_config: Optional[genai.types.GenerationConfigDict] = None,
        safety_settings: Optional[genai.types.SafetySettingDict] = None,
        callback_manager: Optional[CallbackManager] = None,
        api_base: Optional[str] = None,
        transport: Optional[str] = None,
        model_name: Optional[str] = None,
        default_headers: Optional[Dict[str, str]] = None,
        request_options: Optional[genai.types.RequestOptions] = None,
        rate_limit: Optional[int] = 10,  # Default rate limit
        period: Optional[int] = 60,  # Default period
        **generate_kwargs: Any,
    ):
        """Creates a new Gemini model interface."""
        super().__init__(
            api_key=api_key,
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            generation_config=generation_config,
            safety_settings=safety_settings,
            callback_manager=callback_manager,
            api_base=api_base,
            transport=transport,
            model_name=model_name,
            default_headers=default_headers,
            request_options=request_options,
            **generate_kwargs,
        )
        self.rate_limit = rate_limit
        self.period = period
        self.limiter = AsyncLimiter(self.rate_limit, self.period)

    async def achat(
        self, messages: Sequence[ChatMessage], **kwargs: Any
    ) -> ChatResponse:
        request_options = self._request_options or kwargs.pop("request_options", None)
        merged_messages = merge_neighboring_same_role_messages(messages)
        *history, next_msg = map(chat_message_to_gemini, merged_messages)
        chat = self._model.start_chat(history=history)

        async with self.limiter:
            response = await chat.send_message_async(
                next_msg, request_options=request_options, **kwargs
            )
            return chat_from_gemini_response(response)

    async def acomplete(
        self, prompt: str, formatted: bool = False, **kwargs: Any
    ) -> CompletionResponse:
        request_options = self._request_options or kwargs.pop("request_options", None)
        async with self.limiter:
            result = await self._model.generate_content_async(
                prompt, request_options=request_options, **kwargs
            )
            return completion_from_gemini_response(result)

    @dispatcher.span
    async def astructured_predict(
        self,
        output_cls: Type[BaseModel],
        prompt: PromptTemplate,
        llm_kwargs: Optional[Dict[str, Any]] = None,
        **prompt_args: Any,
    ) -> BaseModel:
        r"""Async Structured predict.

        Args:
            output_cls (BaseModel):
                Output class to use for structured prediction.
            prompt (PromptTemplate):
                Prompt template to use for structured prediction.
            llm_kwargs (Optional[Dict[str, Any]]):
                Arguments that are passed down to the LLM invoked by the program.
            prompt_args (Any):
                Additional arguments to format the prompt with.

        Returns:
            BaseModel: The structured prediction output.

        Examples:
            ```python
            from pydantic import BaseModel

            class Test(BaseModel):
                \"\"\"My test class.\"\"\"
                name: str

            from llama_index.core.prompts import PromptTemplate

            prompt = PromptTemplate("Please predict a Test with a random name related to {topic}.")
            output = await llm.astructured_predict(Test, prompt, topic="cats")
            print(output.name)
            ```
        """
        from llama_index.core.program.utils import get_program_for_llm

        dispatcher.event(
            LLMStructuredPredictStartEvent(
                output_cls=output_cls, template=prompt, template_args=prompt_args
            )
        )

        program = get_program_for_llm(
            output_cls,
            prompt,
            self,
            pydantic_program_mode=self.pydantic_program_mode,
        )

        async with self.limiter:
          result = await program.acall(llm_kwargs=llm_kwargs, **prompt_args)
          dispatcher.event(LLMStructuredPredictEndEvent(output=result))
          return result

In [ ]:
Settings.llm = GeminiWithRateLimits(
    # model="models/gemini-1.5-flash-latest",
    api_key=userdata.get("GOOGLE_API_KEY"),
    temperature=1,
    rate_limit=5,
    period=60
)

In [ ]:
# Settings.llm = Gemini(
#     # model="models/gemini-1.5-flash-latest",
#     api_key=userdata.get("GOOGLE_API_KEY"),
#     temperature=1,
#     # rate_limit=5,
#     # period=60
# )

In [ ]:
Settings.llm

GeminiWithRateLimits(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7fada0d56f10>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x7fadb7950ae0>, completion_to_prompt=<function default_completion_to_prompt at 0x7fadb786c4a0>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, model='models/gemini-2.0-flash-001', temperature=1.0, max_tokens=8192, generate_kwargs={}, rate_limit=5, period=60, limiter=<AsyncLimiter(max_rate=5, time_period=60) at 0x7fadb219fbd0 [level: 0.000000, waiters: 0]>)

# Set embedding models

In [ ]:
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
# Use for Gemini Embeddings
# Settings.embed_model = GeminiEmbedding(
#     embed_batch_size = 25,
#     model_name = "models/text-embedding-004",
#     api_key = userdata.get("GOOGLE_API_KEY")
# )

In [ ]:
# Use for HuggingFace Embeddings
Settings.embed_model = HuggingFaceEmbedding(
    model_name = "mixedbread-ai/mxbai-embed-large-v1",
    embed_batch_size = 25,
)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1


Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1


INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']


2 prompts are loaded, with the keys: ['query', 'text']


In [ ]:
print(Settings.embed_model)

model_name='mixedbread-ai/mxbai-embed-large-v1' embed_batch_size=25 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7fada0d56f10> num_workers=None max_length=512 normalize=True query_instruction=None text_instruction=None cache_folder=None


# Parse docs to get the user-level module stuff only

In [ ]:
import asyncio
import re
from typing import List, Optional, Dict, Set

from llama_index.readers.web import BeautifulSoupWebReader
from llama_index.tools.duckduckgo import DuckDuckGoSearchToolSpec
from llama_index.core.prompts import PromptTemplate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.llms import ChatMessage
from llama_index.core import Settings
from collections import defaultdict
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin

ddgs = DuckDuckGoSearchToolSpec()

async def call_model(messages: List[ChatMessage]):
    return await Settings.llm.achat(messages)

async def extract_structure_from_page(
    url: str, nodes, prompt_template, llm
) -> Dict[str, Set[str]]:
    """
    Extracts the structure from the page by using LLM.
    Processes each node from the text splitter and aggregates the results into a dictionary of sets.
    """
    results: Dict[str, Set[str]] = defaultdict(set)

    messages = [
        ChatMessage(
            role="system",
            content=prompt_template.template,
        )
    ]

    for node in nodes:
        messages.append(ChatMessage(role="user", content=node.get_content()))

        response = await call_model(messages)

        entities_by_type = parse_llm_response(response.message.content)

        for entity_type, entities in entities_by_type.items():
            results[entity_type].update(entities)  # Update the sets

        messages.append(response.message)

    return results

def parse_llm_response(response_content: str) -> Dict[str, Set[str]]:
    """
    Parses the LLM response content and extracts entities into a dictionary of sets based on their type.
    """
    entities_by_type: Dict[str, Set[str]] = defaultdict(set)
    lines = response_content.split("\n")

    for line in lines:
        line = line.strip()
        if not line:
            continue

        parts = line.split(":", 1)  # Split on the first colon only
        if len(parts) == 2:
            entity_type = parts[0].strip().lower()
            entity = parts[1].strip()

            if entity_type in ["module", "class"]:
                entities_by_type[entity_type].add(entity)

    return entities_by_type

# Define the function to recursively parse the website
async def recursive_parse_website(
    start_url: str,
    base_path: str,
    max_depth: int | None,
    llm,
    visited_urls=None,
    current_depth=0,
) -> Dict[str, Set[str]]:
    """Recursively parses a website and extracts information."""

    if visited_urls is None:
        visited_urls = set()

    if (
        max_depth is not None
        and current_depth > max_depth
        or start_url in visited_urls
    ):
        return defaultdict(set)

    visited_urls.add(start_url)
    print(f"Processing {start_url} at depth {current_depth}")
    results: Dict[str, Set[str]] = defaultdict(set)

    prompt_template = PromptTemplate(
        """
        You are tasked with extracting structured information from Python module documentation.
        Analyze the given content and identify all the classes and modules, along with their full paths.
        If an entity is clearly a class or module, categorize it as 'class' or 'module' respectively.
        If there is any accompanying code available, please use that to identify the entities.
        The entities may not be displayed with their full path. Smartly find the full paths for the entities in these cases.
        Return each entity on a new line, prefixed with its type followed by a colon, and then the entity's full path or name.

        Example:
        class: module.submodule.ClassName
        module: module

        Page Content:
        {text}
        """
    )

    try:
        response = requests.get(start_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        loader = BeautifulSoupWebReader()
        documents = loader.load_data(urls=[start_url])

        if not documents:
            return defaultdict(set)

        chunk_size = Settings.llm.max_tokens

        splitter = SentenceSplitter(
            chunk_size=chunk_size,
            chunk_overlap=int(0.02 * chunk_size),
        )
        nodes = splitter.get_nodes_from_documents(documents)

        extracted_info = await extract_structure_from_page(
            start_url, nodes, prompt_template, llm
        )
        results: Dict[str, Set[str]] = defaultdict(set, extracted_info)

        links = set()
        for link in soup.find_all("a", href=True):
            href = link.get("href")
            if "#" in href:
              href = href.split("#")[0]  # Remove anchor part
            if not href:
              continue

            absolute_url = urljoin(start_url, href)  # Construct absolute URL

            if absolute_url.startswith(base_path):
                links.add(absolute_url)

        # Recursively process links
        sub_results = await asyncio.gather(
            *[
                recursive_parse_website(
                    url, base_path, max_depth, llm, visited_urls, current_depth + 1
                )
                for url in links
            ]
        )

        for sub_result in sub_results:
            for key, value in sub_result.items():
                results[key].update(value)

    except Exception as e:
        print(f"Could not get links from page {start_url}: {e}")
        return results

    return results

async def get_api_reference_for_module(
    module_name: str, llm
) -> tuple[str, str]:
    """
    Finds the API reference for a given Python module using DuckDuckGo search,
    selects the most relevant link using the LLM, and then determines the base path
    for recursive parsing. Returns both the most relevant URL and the base path.
    """

    query = f"{module_name} Python API Reference"
    search_results = ddgs.duckduckgo_full_search(query=query)

    # Construct a prompt for the LLM to select the best link
    search_results_text = ""
    for i, result in enumerate(search_results):
        search_results_text += f"{i+1}. {result['title']} ({result['href']})\n"

    prompt_template = PromptTemplate(
        """
        You are given a list of search results for the query: "{query}".
        Please select the URL that is most likely to contain the API reference documentation
        for the Python module.

        Additionally, analyze the selected URL and determine a suitable base path for a website crawler.
        The base path maybe a prefix of the selected URL or the selected URL itself that points to a section
        of the website likely to contain API reference or documentation pages. Consider common URL structures
        and keywords like 'api', 'api_reference', 'reference', 'apireference', or 'docs'.

        Keep in mind that you may return the selected URL as the base path if it looks like a valid base path.

        Return the output in the format:
        <selected_url>
        <base_path>

        Search Results:
        {search_results_text}

        Most relevant URL and base path:
        """
    )

    # Get the LLM's selection
    response = (
        await llm.apredict(
            prompt_template, query=query, search_results_text=search_results_text
        )
    ).strip()
    try:
        selected_url, base_path = response.split("\n")
    except ValueError:
        print(f"Could not get url and base path from llm response: {response}")
        return "", ""

    return selected_url, base_path

async def extract_user_level_functionalities_for_module(
    module_name: str, llm, max_depth: Optional[int] = 2
) -> Dict[str, Set[str]]:
    """
    This is a wrapper function that finds the user level functionalities for a given module by:
    1. Finding the API reference URL and base path using `get_api_reference_for_module`.
    2. Recursively parsing the website using `recursive_parse_website`.

    Returns:
        A dictionary of sets, where keys are entity types (module, class, method/function) and values are sets of corresponding entities.
    """
    selected_url, base_path = await get_api_reference_for_module(module_name, llm)

    if not selected_url or not base_path:
        print(f"Could not find API reference for module: {module_name}")
        return defaultdict(set)

    print(f"Extracting information from: {selected_url} (Base path: {base_path})")

    results = await recursive_parse_website(
        selected_url, base_path, max_depth, llm
    )

    return results

In [ ]:
#@title Example Usage of DDGS for Finding API Reference links for Python modules
ddgs.duckduckgo_full_search("pydantic Python API Reference")

[{'title': 'Welcome to Pydantic - Pydantic',
  'href': 'https://docs.pydantic.dev/latest/',
  'body': 'Pydantic¶ Documentation for version: v2.10.5. Pydantic is the most widely used data validation library for Python. Fast and extensible, Pydantic plays nicely with your linters/IDE/brain. Define how data should be in pure, canonical Python 3.8+; validate it with Pydantic.'},
 {'title': 'Web and API Requests - Pydantic',
  'href': 'https://docs.pydantic.dev/latest/examples/requests/',
  'body': "Pydantic is instrumental in many web frameworks and libraries, such as FastAPI, Django, Flask, and HTTPX. httpx requests¶ httpx is a HTTP client for Python 3 with synchronous and asynchronous APIs. In the below example, we query the JSONPlaceholder API to get a user's data and validate it with a Pydantic model."},
 {'title': 'python - FastAPI / Pydantic circular references in separate files ...',
  'href': 'https://stackoverflow.com/questions/63420889/fastapi-pydantic-circular-references-in-sepa

In [ ]:
#@title Example Usage of get_api_reference_for_module
module = "requests"
url, base_path = await get_api_reference_for_module(module, llm = Settings.llm)
print(f"URL: {url}")
print(f"Base Path: {base_path}")

URL: https://docs.python-requests.org/en/latest/api/
Base Path: https://docs.python-requests.org/en/latest/api/


In [ ]:
module_name = "requests"  # Example module
max_depth = None  # You can adjust the depth

extracted_info = await extract_user_level_functionalities_for_module(
    module_name, Settings.llm, max_depth
)

Extracting information from: https://docs.python-requests.org/en/latest/api/ (Base path: https://docs.python-requests.org/en/latest/api/)
Processing https://docs.python-requests.org/en/latest/api/ at depth 0


In [ ]:
print("Number of extracted modules:", len(extracted_info["module"]))
print("Number of extracted classes:", len(extracted_info["class"]))

Number of extracted modules: 5
Number of extracted classes: 21


In [ ]:
extracted_info["module"]

{'requests',
 'requests.adapters',
 'requests.auth',
 'requests.cookies',
 'requests.utils'}

Verify the above results like so:


1.   Head over to the [requests API Reference](https://docs.python-requests.org/en/latest/api/).
2.   Use the following code to find the number of methods/functions: (DEPRECATED)


```javascript
const elements = document.querySelectorAll('.py.method, .py.function');

// Convert the NodeList to an array
const elementsArray = Array.from(elements);
console.log(elementsArray.length); // 94
```

3.   Use the following code to find the number of classes (exceptions included):


```javascript
const elements = document.querySelectorAll('.py.class, .py.exception');

// Convert the NodeList to an array
const elementsArray = Array.from(elements);
console.log(elementsArray.length); // 21
```






## Testing extract_user_level_functionalities_for_module with pydantic

In [ ]:
module_name = "pydantic"  # Example module
max_depth = None  # You can adjust the depth

extracted_info = await extract_user_level_functionalities_for_module(
    module_name, Settings.llm, max_depth
)

Extracting information from: https://docs.pydantic.dev/latest/api/base_model/ (Base path: https://docs.pydantic.dev/latest/api/)
Processing https://docs.pydantic.dev/latest/api/base_model/ at depth 0
Processing https://docs.pydantic.dev/latest/api/functional_serializers/ at depth 1
Processing https://docs.pydantic.dev/latest/api/config/ at depth 1
Processing https://docs.pydantic.dev/latest/api/functional_validators/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_extra_types_color/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_extra_types_phone_numbers/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_extra_types_script_code/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_extra_types_pendulum_dt/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_settings/ at depth 1
Processing https://docs.pydantic.dev/latest/api/json_schema/ at depth 1
Processing https://docs.pydantic.dev/latest/api/ex

In [ ]:
print("Number of extracted modules:", len(extracted_info["module"]))
print("Number of extracted classes:", len(extracted_info["class"]))
# print("Number of extracted methods/functions:", len(extracted_info["method/function"]))

Number of extracted modules: 109
Number of extracted classes: 384


## Testing extract_user_level_functionalities_for_module with bs4

In [ ]:
module_name = "BeautifulSoup4"  # Example module
max_depth = None  # You can adjust the depth

extracted_info = await extract_user_level_functionalities_for_module(
    module_name, Settings.llm, max_depth
)

Extracting information from: https://www.crummy.com/software/BeautifulSoup/bs4/doc/ (Base path: https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/ at depth 0
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/genindex.html at depth 1
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/py-modindex.html at depth 1
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/_sources/index.rst.txt at depth 1
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/index.html at depth 2
Could not get links from page https://www.crummy.com/software/BeautifulSoup/bs4/doc/index.html: 429 Resource has been exhausted (e.g. check quota).


In [ ]:
print("Number of extracted modules:", len(extracted_info["module"]))
print("Number of extracted classes:", len(extracted_info["class"]))
# print("Number of extracted methods/functions:", len(extracted_info["method/function"]))

Number of extracted modules: 1
Number of extracted classes: 16


# Use `inspect` to extract MODULES, CLASSES and METHODS in the user specified python module

## Use following code with information extracted from DuckDuckGo Search usage

In [ ]:
import inspect
import importlib
from typing import Any, Dict, List, Tuple, Union, Set


def safe_get_source(obj) -> str:
    """Retrieve source code or fallback to repr."""
    try:
        return inspect.getsource(obj)
    except (OSError, TypeError):
        return f"# Representation:\n{repr(obj)}"


def process_member(
    name: str,
    obj: Any,
    parent_entity: Dict[str, Any],
    current_path: str,
    extracted_info: Dict[str, Set[str]],
    parsed_entities: Set[str],
) -> Tuple[List[Dict], List[Dict]]:
    """
    Process individual members of a class or module.

    Args:
        name (str): Name of the member.
        obj (Any): Member object.
        parent_entity (Dict[str, Any]): Parent entity.
        current_path (str): Current traversal path.
        extracted_info (Dict[str, Set[str]]): Extracted info from website parsing.
        parsed_entities (Set[str]): Set of already parsed entities.

    Returns:
        Tuple[List[Dict], List[Dict]]:
            - List of entities
            - List of relations
    """
    entities, relations = [], []
    obj_path = f"{current_path}.{name}"

    # Skip if name starts with '_'
    if name.startswith("_"):
        return [], []

    if inspect.ismodule(obj) or inspect.isclass(obj):
        if obj_path not in extracted_info["module"] and obj_path not in extracted_info["class"]:
            return [], []

        if obj_path in parsed_entities:
            return [], []

        # Recurse into modules or nested classes
        sub_entities, sub_relations = traverse_recursive(
            obj, obj_path, extracted_info, parsed_entities
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

        # Create appropriate hierarchical relation
        relation_type = "HAS_MODULE" if inspect.ismodule(obj) else "HAS_CLASS"
        relations.append(create_relation(relation_type, parent_entity, entities[0]))

    elif inspect.isfunction(obj) or inspect.ismethod(obj):
        # Process function or method
        source_code = safe_get_source(obj)
        method_entity = create_entity(
            "METHOD",
            obj_path,
            {
                "source": source_code,
                # "signature": signature,
            },
        )
        entities.append(method_entity)
        if parent_entity:
            relations.append(create_relation("HAS_METHOD", parent_entity, method_entity))

    return entities, relations


def traverse_recursive(
    obj: Any,
    current_path: str,
    extracted_info: Dict[str, Set[str]],
    parsed_entities: Set[str],
) -> Tuple[List[Dict], List[Dict]]:
    """
    Recursively traverse an object (module or class).

    Args:
        obj (Any): The object to traverse.
        current_path (str): The current traversal path.
        extracted_info (Dict[str, Set[str]]): Extracted info from website parsing.
        parsed_entities (Set[str]): Set of already parsed entities.

    Returns:
        Tuple[List[Dict], List[Dict]]:
            - List of entities
            - List of relations.
    """
    entities, relations = [], []
    obj_name = getattr(obj, "__name__", str(obj))
    entity_type = "MODULE" if inspect.ismodule(obj) else "CLASS"

    # Check if this module/class has already been parsed
    if current_path in parsed_entities:
        return [], []

    # Skip if not in extracted_info
    if (
        entity_type == "MODULE"
        and current_path not in extracted_info["module"]
    ) or (
        entity_type == "CLASS"
        and current_path not in extracted_info["class"]
    ):
        return [], []

    # Create entity for the module or class
    source = safe_get_source(obj)
    properties = {"source": source}

    entity = create_entity(entity_type, current_path, properties)
    entities.append(entity)
    parsed_entities.add(current_path)  # Mark this entity as parsed

    # Handle inheritance for classes
    if inspect.isclass(obj):
        for base in obj.__bases__:
            if base.__name__ not in ["object", "type"]:
                base_path = f"{base.__module__}.{base.__name__}"
                if base_path in extracted_info["class"] or base_path in extracted_info["module"]:
                    base_entity = create_entity(
                        "CLASS",
                        base_path,
                        {"source": safe_get_source(base)},
                    )
                    entities.append(base_entity)
                    relations.append(create_relation("HAS_BASECLASS", entity, base_entity))

    # Traverse and process members
    for name, member in inspect.getmembers(obj):
        sub_entities, sub_relations = process_member(
            name, member, entity, current_path, extracted_info, parsed_entities
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    return entities, relations


def process_module_recursive(
    module_name_or_object: Union[str, Any], extracted_info: Dict[str, Set[str]]
):
    """
    Process a module or object recursively, traversing only extracted modules/classes.

    Args:
        module_name_or_object (Union[str, Any]): Module name or object.
        extracted_info (Dict[str, Set[str]]): Extracted info from website parsing.
    """
    parsed_entities = set()
    entities, relations = [], []

    # Sort by path length to process shorter paths first (parents before children)
    for item_path in sorted(
        list(extracted_info["module"]) + list(extracted_info["class"]),
        key=len,
    ):
        if item_path in parsed_entities:
            continue  # Skip if already parsed

        try:
            if isinstance(module_name_or_object, str):
                if "." in item_path:
                    module_name, object_name = item_path.rsplit(".", 1)
                    module = importlib.import_module(module_name)
                    module_or_object = getattr(module, object_name)
                else:
                    module_or_object = importlib.import_module(item_path)
            else:
                module_or_object = module_name_or_object

            sub_entities, sub_relations = traverse_recursive(
                module_or_object, item_path, extracted_info, parsed_entities
            )
            entities.extend(sub_entities)
            relations.extend(sub_relations)
        except (ImportError, AttributeError) as e:
            print(f"Error processing {item_path}: {e}")

    print(f"Processed {len(entities)} entities and {len(relations)} relations.")

    return entities, relations


## Use following code without information extracted from DuckDuckGo Search usage

In [ ]:
"""
ReflEx (Reflection and Extraction) Module

This module provides utilities to extract entities and relationships from Python modules and classes, such as:
- Modules
- Classes
- Methods
- Inheritance
- Imports
"""

from typing import Any, Dict, List, Set, Tuple
import inspect
import importlib
import re
import subprocess

class Entity:
    """
    Module-agnostic entity class.
    """
    def __init__(self, name: str, label: str, properties: Dict):
        self.name = name
        self.label = label
        self.properties = properties

    def __repr__(self):
        return f"{self.label} ({self.name})"

class Relationship:
    """
    Module-agnostic relation class.
    """
    def __init__(self, from_entity: Entity, to_entity: Entity, label: str, properties: Dict):
        self.from_entity = from_entity
        self.to_entity = to_entity
        self.label = label
        self.properties = properties

    def __repr__(self):
        return f"{self.from_entity} --{self.label}--> {self.to_entity}"

def get_unused_imports(module_name: str, imported_names: Set[str]) -> Set[str]:
    """
    Get the list of unused imports in a module using flake8, returning the full paths.

    Args:
        module_name (str): The name of the module.
        imported_names (Set[str]): The set of fully qualified imported names.

    Returns:
        Set[str]: The set of unused imports (full paths).
    """
    try:
        # Get the full file path of the module
        module = importlib.import_module(module_name)
        module_file_path = module.__file__

        # Run flake8 and capture output
        process = subprocess.run(
            ["flake8", "--select", "F401", module_file_path],
            capture_output=True,
            text=True,
            check=False,
        )

        output = process.stdout
        unused_names = list()

        # Parse the output for unused import errors (F401), extracting the simple names
        for line in output.splitlines():
            match = re.search(r":\d+:(\d+): F401 '(.*)' imported but unused", line)
            if match:
                unused_names.append(match.group(2))


        # Handle aliases and from-imports statements
        for i in range(len(unused_names)):
            name = unused_names[i].strip()
            if " as " in name:
                unused_names[i] = name.split(" as ")[0]
            if name.startswith("from "):
                unused_names[i] = name.split("from ")[1]
            if " import " in name:
                # Import this unused name to get the fully qualified name
                split_name = name.split(" import ")
                module_name = split_name[0]
                imported_name = split_name[1]
                imported_module = importlib.import_module(module_name)
                imported_entity = getattr(imported_module, imported_name)
                unused_names[i] = get_fully_qualified_name(imported_entity)

        unused_imports = set()
        for full_name in imported_names:
            # if any(name == full_name or full_name.startswith(f"{name}.") for name in unused_names):
            #     unused_imports.add(full_name)
            if any(full_name.endswith(name) for name in unused_names):
                 unused_imports.add(full_name)

        return unused_imports
    except Exception as e:
        print("error:", e)
        return set()

def get_fully_qualified_name(cls):
    """
    Get the fully qualified name of a class.

    Args:
        cls: The class object.

    Returns:
        str: The fully qualified name of the class.
    """
    return f"{cls.__module__}.{cls.__qualname__}"

def check_dunders(path):
    """
    Check if a path starts with an underscore or contains '._'.

    Args:
        path (str): The path to check.

    Returns:
        bool: True if the path starts with an underscore or contains '._', False otherwise.
    """
    return path.startswith("_") or "._" in path

def safe_get_source(obj) -> str:
    """
    Retrieve the source code of an object or fallback to its representation.

    Args:
        obj: The object to retrieve the source code for.

    Returns:
        str: The source code or representation of the object.
    """
    try:
        return inspect.getsource(obj)
    except (OSError, TypeError):
        return f"# Representation:\n{repr(obj)}"

def process_member(
    name: str,
    obj: Any,
    root_module_path: str,
    parent_entity,
    processed_paths: Set[str],
    other_relations: Dict[str, List[Tuple]] = None,
) -> Tuple[List[Entity], List[Relationship]]:
    """
    Process individual members of a class or module.

    Args:
        name (str): Name of the member.
        obj (Any): Member object.
        root_module_path (str): The root module path.
        parent_entity: Parent entity.
        processed_paths (Set[str]): Set of processed paths.
        other_relations (Dict[str, List[Tuple]], optional): Dictionary to store other relations.

    Returns:
        Tuple[List[Entity], List[Relationship]]: List of entities and list of relations.
    """
    entities, relations = [], []

    if name.startswith("_"):
        return [], []

    if inspect.ismodule(obj) or inspect.isclass(obj):
        obj_path = None

        if inspect.ismodule(obj):
            obj_path = obj.__name__
        elif inspect.isclass(obj):
            obj_path = get_fully_qualified_name(obj)

        if not obj_path.startswith(root_module_path):
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        if obj_path in processed_paths:
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        if not obj_path.startswith(parent_entity.name):
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        if check_dunders(obj_path):
            return [], []

        sub_entities, sub_relations = traverse_recursive(
            obj, root_module_path, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

        relation_type = "HAS_SUBMODULE" if inspect.ismodule(obj) else "HAS_CLASS"
        relations.append(Relationship(parent_entity, entities[0], relation_type, {}))

    elif inspect.isfunction(obj) and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
        obj_path = get_fully_qualified_name(obj)

        if check_dunders(obj_path):
            return [], []

        if not obj_path.startswith(root_module_path):
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        if obj_path in processed_paths:
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        elif not obj_path.startswith(parent_entity.name):
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        else:
            processed_paths.add(obj_path)

        source_code = safe_get_source(obj)
        function_entity = Entity(
            obj_path,
            "FUNCTION",
            {
                "source": source_code,
                "__name__": obj.__name__
            }
        )
        entities.append(function_entity)
        if parent_entity:
            relations.append(Relationship(parent_entity, function_entity, "HAS_FUNCTION", {}))

    elif parent_entity.label == "CLASS" and (inspect.ismethod(obj) or inspect.isfunction(obj)):
        obj_path = get_fully_qualified_name(obj)

        if check_dunders(obj_path):
            return [], []

        if not obj_path.startswith(root_module_path):
            return [], []

        if obj_path in processed_paths:
            return [], []
        else:
            processed_paths.add(obj_path)

        source_code = safe_get_source(obj)
        method_entity = Entity(
            obj_path,
            "METHOD",
            {
                "source": source_code,
                "__name__": obj.__name__
            }
        )
        entities.append(method_entity)
        if parent_entity:
            relations.append(Relationship(parent_entity, method_entity, "HAS_METHOD", {}))

    return entities, relations

def traverse_recursive(
    obj: Any,
    root_module_path: str,
    processed_paths: Set[str],
    other_relations: Dict[str, List[Tuple]] = None,
) -> Tuple[List[Entity], List[Relationship]]:
    """
    Recursively traverse an object (module or class).

    Args:
        obj (Any): The object to traverse.
        root_module_path (str): The root module path.
        processed_paths (Set[str]): Set of processed paths.
        other_relations (Dict[str, List[Tuple]], optional): Dictionary to store other relations.

    Returns:
        Tuple[List[Entity], List[Relation]]: List of entities and list of relations.
    """
    entities, relations = [], []
    obj_path = None
    entity_type = "MODULE" if inspect.ismodule(obj) else "CLASS"
    if entity_type == "MODULE":
        obj_path = obj.__name__
    elif entity_type == "CLASS":
        obj_path = get_fully_qualified_name(obj)

    if not obj_path.startswith(root_module_path):
        return [], []

    if obj_path in processed_paths:
        return [], []

    else:
        processed_paths.add(obj_path)

    source = safe_get_source(obj)
    properties = {
        "source": source,
        "__name__": obj.__name__,
    }

    entity = Entity(obj_path, entity_type, properties)
    entities.append(entity)

    if inspect.isclass(obj):
        for base in obj.__bases__:
            if base.__name__ not in ["object", "type"]:
                base_path = f"{base.__module__}.{base.__name__}"
                other_relations["INHERITS"].add((obj_path, base_path))

    # Process sub-modules
    for name, member in inspect.getmembers(obj, inspect.ismodule):
        sub_entities, sub_relations = process_member(
            name, member, root_module_path, entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process sub-classes
    for name, member in inspect.getmembers(obj, inspect.isclass):
        sub_entities, sub_relations = process_member(
            name, member, root_module_path, entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process functions
    for name, member in inspect.getmembers(obj, inspect.isfunction):
        sub_entities, sub_relations = process_member(
            name, member, root_module_path, entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process methods
    for name, member in inspect.getmembers(obj, inspect.ismethod):
        sub_entities, sub_relations = process_member(
            name, member, root_module_path, entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    return entities, relations

def process_module(module_name: str):
    """
    Process a module or object recursively.

    Args:
        module_name (str): Module name or object.

    Returns:
        Tuple[List[Entity], List[Relation], Dict]: List of entities, list of relations, and other relations.
    """
    entities, relations = [], []
    processed_paths = set()
    other_relations = {"INHERITS": set(), "IMPORTS": set(), "EXPOSES": set()}

    module_object = importlib.import_module(module_name)

    source = safe_get_source(module_object)
    properties = {
        "source": source,
        "__name__": module_object.__name__,
    }

    root_entity = Entity(module_name, "ROOT_MODULE", properties)
    entities.append(root_entity)

    # Process sub-modules
    for name, member in inspect.getmembers(module_object, inspect.ismodule):
        sub_entities, sub_relations = process_member(
            name, member, module_name, root_entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process sub-classes
    for name, member in inspect.getmembers(module_object, inspect.isclass):
        sub_entities, sub_relations = process_member(
            name, member, module_name, root_entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process functions
    for name, member in inspect.getmembers(module_object, inspect.isfunction):
        sub_entities, sub_relations = process_member(
            name, member, module_name, root_entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process methods
    for name, member in inspect.getmembers(module_object, inspect.ismethod):
        sub_entities, sub_relations = process_member(
            name, member, module_name, root_entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Convert IMPORTS to EXPOSES for unused imports
    imported_relations = other_relations["IMPORTS"]
    updated_imported_relations = set()
    module_to_imports = {}

    for from_entity, to_entity in imported_relations:
        full_module_path = from_entity

        if full_module_path not in module_to_imports:
            module_to_imports[full_module_path] = set()

        module_to_imports[full_module_path].add(to_entity)

    for module, imports in module_to_imports.items():
        unused_imports = get_unused_imports(module, imports)
        used_imports = set(imports) - set(unused_imports)

        for unused_import in unused_imports:
            other_relations["EXPOSES"].add((module, unused_import))

        for used_import in used_imports:
            updated_imported_relations.add((module, used_import))

    other_relations["IMPORTS"] = updated_imported_relations

    cleaned_exposed_relations = set()
    cleaned_imports_relations = set()

    # Clean up IMPORTS relations
    for from_entity, to_entity in other_relations["IMPORTS"]:
        to_entity_ancestor_path = ".".join(to_entity.split(".")[:-1])
        if to_entity_ancestor_path == from_entity:
            continue
        elif from_entity == module_name and to_entity.startswith(module_name):
            other_relations["EXPOSES"].add((from_entity, to_entity))
        else:
            cleaned_imports_relations.add((from_entity, to_entity))

    # Clean up EXPOSED relations
    for from_entity, to_entity in other_relations["EXPOSES"]:
        to_entity_ancestor_path = ".".join(to_entity.split(".")[:-1])
        if to_entity_ancestor_path == from_entity:
            continue
        if not to_entity.startswith(from_entity):
            continue
        cleaned_exposed_relations.add((from_entity, to_entity))

    other_relations["EXPOSES"] = list(cleaned_exposed_relations)
    other_relations["IMPORTS"] = list(cleaned_imports_relations)
    other_relations["INHERITS"] = list(other_relations["INHERITS"])

    # Deduplicate entities and relations
    entities = list(set(entities))
    relations = list(set(relations))

    print(f"Processed {len(entities)} entities and {len(relations)} relations in {module_name}.")

    return entities, relations, other_relations

## Try a demo module

In [ ]:
!pip install aerosandbox

In [ ]:
entities, relations, other_relations = process_module("tkinter")
# entities, relations = process_module_recursive("bs4", extracted_info)

Processed 545 entities and 544 relations in tkinter.


### Check entities processed

In [ ]:
def print_modules(entities):
  for entity in entities:
    if entity.label == "MODULE":
      print(entity.label, entity.name)

def print_classes(entities):
  for entity in entities:
    if entity.label == "CLASS":
      print(entity.label, entity.name)

def print_methods(entities):
  for entity in entities:
    if entity.label == "METHOD":
      print(entity.label, entity.name)

def print_functions(entities):
  for entity in entities:
    if entity.label == "FUNCTION":
      print(entity.label, entity.name)

In [ ]:
print_modules(entities)

MODULE requests.api
MODULE requests.models
MODULE requests.exceptions
MODULE requests.structures
MODULE requests.adapters
MODULE requests.status_codes
MODULE requests.utils
MODULE requests.certs
MODULE requests.cookies
MODULE requests.auth
MODULE requests.packages
MODULE requests.sessions
MODULE requests.compat
MODULE requests.hooks


In [ ]:
print_classes(entities)

CLASS requests.exceptions.MissingSchema
CLASS requests.exceptions.InvalidHeader
CLASS requests.cookies.MockRequest
CLASS requests.exceptions.InvalidProxyURL
CLASS requests.cookies.CookieConflictError
CLASS requests.models.RequestEncodingMixin
CLASS requests.models.Request
CLASS requests.exceptions.StreamConsumedError
CLASS requests.structures.LookupDict
CLASS requests.exceptions.FileModeWarning
CLASS requests.exceptions.TooManyRedirects
CLASS requests.exceptions.UnrewindableBodyError
CLASS requests.adapters.BaseAdapter
CLASS requests.structures.CaseInsensitiveDict
CLASS requests.models.PreparedRequest
CLASS requests.exceptions.URLRequired
CLASS requests.models.RequestHooksMixin
CLASS requests.exceptions.InvalidJSONError
CLASS requests.sessions.Session
CLASS requests.exceptions.HTTPError
CLASS requests.models.Response
CLASS requests.exceptions.SSLError
CLASS requests.exceptions.RequestsDependencyWarning
CLASS requests.exceptions.InvalidURL
CLASS requests.auth.HTTPDigestAuth
CLASS reques

In [ ]:
print_methods(entities)

METHOD requests.adapters.HTTPAdapter.build_response
METHOD requests.models.Response.iter_content
METHOD requests.cookies.MockRequest.get_full_url
METHOD requests.cookies.RequestsCookieJar.set
METHOD requests.models.PreparedRequest.prepare_body
METHOD requests.adapters.HTTPAdapter.proxy_manager_for
METHOD requests.cookies.RequestsCookieJar.values
METHOD requests.sessions.Session.delete
METHOD requests.cookies.MockRequest.get_host
METHOD requests.cookies.MockRequest.is_unverifiable
METHOD requests.models.PreparedRequest.prepare_content_length
METHOD requests.sessions.Session.patch
METHOD requests.models.PreparedRequest.prepare_url
METHOD requests.sessions.SessionRedirectMixin.get_redirect_target
METHOD requests.sessions.Session.send
METHOD requests.models.RequestHooksMixin.deregister_hook
METHOD requests.models.PreparedRequest.prepare_cookies
METHOD requests.adapters.BaseAdapter.close
METHOD requests.adapters.HTTPAdapter.get_connection
METHOD requests.models.RequestHooksMixin.register_ho

In [ ]:
print_functions(entities)

FUNCTION requests.utils.check_header_validity
FUNCTION requests.utils.urldefragauth
FUNCTION requests.api.post
FUNCTION requests.utils.default_headers
FUNCTION requests.api.options
FUNCTION requests.hooks.default_hooks
FUNCTION requests.cookies.morsel_to_cookie
FUNCTION requests.utils.extract_zipped_paths
FUNCTION requests.api.head
FUNCTION requests.utils.get_netrc_auth
FUNCTION requests.utils.set_environ
FUNCTION requests.cookies.create_cookie
FUNCTION requests.utils.dict_from_cookiejar
FUNCTION requests.utils.is_valid_cidr
FUNCTION requests.api.get
FUNCTION requests.hooks.dispatch_hook
FUNCTION requests.cookies.get_cookie_header
FUNCTION requests.sessions.merge_hooks
FUNCTION requests.check_compatibility
FUNCTION requests.utils.from_key_val_list
FUNCTION requests.cookies.extract_cookies_to_jar
FUNCTION requests.utils.rewind_body
FUNCTION requests.api.patch
FUNCTION requests.utils.parse_list_header
FUNCTION requests.api.request
FUNCTION requests.utils.get_environ_proxies
FUNCTION requ

In [ ]:
other_relations["INHERITS"]

[('requests.exceptions.InvalidHeader', 'builtins.ValueError'),
 ('requests.exceptions.SSLError', 'requests.exceptions.ConnectionError'),
 ('requests.exceptions.RequestsWarning', 'builtins.Warning'),
 ('requests.models.PreparedRequest', 'requests.models.RequestEncodingMixin'),
 ('requests.exceptions.TooManyRedirects',
  'requests.exceptions.RequestException'),
 ('requests.cookies.RequestsCookieJar', 'http.cookiejar.CookieJar'),
 ('requests.exceptions.InvalidURL', 'builtins.ValueError'),
 ('requests.exceptions.InvalidURL', 'requests.exceptions.RequestException'),
 ('requests.sessions.Session', 'requests.sessions.SessionRedirectMixin'),
 ('requests.structures.LookupDict', 'builtins.dict'),
 ('requests.models.Request', 'requests.models.RequestHooksMixin'),
 ('requests.exceptions.StreamConsumedError',
  'requests.exceptions.RequestException'),
 ('requests.exceptions.FileModeWarning',
  'requests.exceptions.RequestsWarning'),
 ('requests.auth.HTTPDigestAuth', 'requests.auth.AuthBase'),
 ('re

In [ ]:
other_relations["IMPORTS"]

[('requests.utils', 'requests.certs'),
 ('requests.adapters', 'requests.exceptions.SSLError'),
 ('requests.adapters', 'socket'),
 ('requests.models', 'collections.abc.Mapping'),
 ('requests.sessions', 'requests.models.Request'),
 ('requests.adapters', 'urllib3.poolmanager.proxy_from_url'),
 ('requests.sessions', 'requests.hooks.dispatch_hook'),
 ('requests.models', 'requests.exceptions.HTTPError'),
 ('requests.api', 'requests.sessions'),
 ('requests.models', 'requests.exceptions.JSONDecodeError'),
 ('requests.compat', 'chardet'),
 ('requests.utils', 'builtins.str'),
 ('requests.models', 'http.cookiejar'),
 ('requests.status_codes', 'requests.structures.LookupDict'),
 ('requests.models', 'requests.structures.CaseInsensitiveDict'),
 ('requests.utils', 'struct'),
 ('requests.adapters', 'os'),
 ('requests.models', 'requests.cookies.get_cookie_header'),
 ('requests.models', 'requests.exceptions.InvalidJSONError'),
 ('requests.models', 'collections.abc.Callable'),
 ('requests.utils', 'collec

In [ ]:
other_relations["EXPOSES"]

[('requests', 'requests.exceptions.FileModeWarning'),
 ('requests', 'requests.api.options'),
 ('requests', 'requests.exceptions.ConnectionError'),
 ('requests', 'requests.api.post'),
 ('requests', 'requests.exceptions.RequestException'),
 ('requests', 'requests.models.Request'),
 ('requests', 'requests.exceptions.ConnectTimeout'),
 ('requests', 'requests.api.head'),
 ('requests', 'requests.exceptions.TooManyRedirects'),
 ('requests', 'requests.api.put'),
 ('requests', 'requests.exceptions.HTTPError'),
 ('requests', 'requests.exceptions.RequestsDependencyWarning'),
 ('requests', 'requests.models.Response'),
 ('requests', 'requests.api.get'),
 ('requests', 'requests.exceptions.JSONDecodeError'),
 ('requests', 'requests.api.delete'),
 ('requests', 'requests.sessions.session'),
 ('requests', 'requests.models.PreparedRequest'),
 ('requests', 'requests.exceptions.URLRequired'),
 ('requests', 'requests.sessions.Session'),
 ('requests', 'requests.api.request'),
 ('requests', 'requests.exceptio

### Insert the entities and relations into Neo4j

In [ ]:
pg_store.upsert_to_neo4j("tkinter", entities, relations, other_relations)

# Upload all standard python modules

In [ ]:
standard_modules = ['string', 're', 'difflib', 'textwrap', 'unicodedata', 'stringprep', 'readline', 'rlcompleter', 'struct', 'codecs', 'datetime', 'zoneinfo', 'calendar', 'collections', 'heapq', 'bisect', 'array', 'weakref', 'types', 'copy', 'pprint', 'reprlib', 'enum', 'graphlib', 'numbers', 'math', 'cmath', 'decimal', 'fractions', 'random', 'statistics', 'itertools', 'functools', 'operator', 'pathlib', 'stat', 'filecmp', 'tempfile', 'glob', 'fnmatch', 'linecache', 'shutil', 'pickle', 'copyreg', 'shelve', 'marshal', 'dbm', 'sqlite3', 'zlib', 'gzip', 'bz2', 'lzma', 'zipfile', 'tarfile', 'csv', 'configparser', 'tomllib', 'netrc', 'plistlib', 'hashlib', 'hmac', 'secrets', 'os', 'io', 'time', 'logging', 'platform', 'errno', 'ctypes', 'argparse', 'optparse', 'getpass', 'fileinput', 'curses', 'threading', 'multiprocessing', 'concurrent', 'subprocess', 'sched', 'queue', 'contextvars', 'asyncio', 'socket', 'ssl', 'select', 'selectors', 'signal', 'mmap', 'email', 'json', 'mailbox', 'mimetypes', 'base64', 'binascii', 'quopri', 'html', 'xml', 'pyexpat', 'webbrowser', 'wsgiref', 'urllib', 'http', 'ftplib', 'poplib', 'imaplib', 'smtplib', 'uuid', 'socketserver', 'xmlrpc', 'ipaddress', 'wave', 'colorsys', 'gettext', 'locale', 'turtle', 'cmd', 'shlex', 'tkinter', 'typing', 'pydoc', 'doctest', 'unittest', 'test', 'bdb', 'faulthandler', 'pdb', 'profile', 'timeit', 'trace', 'tracemalloc', 'venv', 'zipapp', 'sys', 'sysconfig', 'builtins', 'warnings', 'dataclasses', 'contextlib', 'abc', 'atexit', 'traceback', 'gc', 'inspect', 'site', 'code', 'codeop', 'zipimport', 'pkgutil', 'modulefinder', 'runpy', 'importlib', 'ast', 'symtable', 'token', 'keyword', 'tokenize', 'tabnanny', 'pyclbr', 'py_compile', 'compileall', 'dis', 'pickletools', 'getopt']

In [ ]:
from tqdm import tqdm

for standard_module in tqdm(standard_modules, desc="Processing Modules"):
    entities, relations, other_relations = process_module(standard_module)
    pg_store.upsert_to_neo4j(standard_module, entities, relations, other_relations)

Processing Modules:   0%|          | 0/163 [00:00<?, ?it/s]

Processed 16 entities and 15 relations in string.


Processing Modules:   1%|          | 1/163 [00:10<27:20, 10.13s/it]

Processed 19 entities and 18 relations in re.


Processing Modules:   1%|          | 2/163 [00:20<27:44, 10.34s/it]

Processed 26 entities and 25 relations in difflib.


... (317 outputs left out)

Processed 31 entities and 30 relations in pickletools.


Processing Modules:  99%|█████████▉| 162/163 [37:27<00:11, 11.97s/it]

Processed 8 entities and 7 relations in getopt.


Processing Modules: 100%|██████████| 163/163 [37:36<00:00, 13.85s/it]


# Test the COMStore

## Test querying bs4.BeautifulSoup using a sample prompt

In [ ]:
prompt = "allows easy extraction, navigation and modification of elements"

In [ ]:
prompt_embedding = Settings.embed_model.get_query_embedding(prompt)
print("Length:", len(prompt_embedding))
print("Prompt Embedding:", prompt_embedding)
print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Length: 1024
Prompt Embedding: [-0.023381048813462257, -0.01772773079574108, -0.009805514477193356, -0.035045262426137924, -0.0173393152654171, 0.03186623007059097, 0.0021248760167509317, -0.007745752576738596, 0.06094205379486084, 0.034395478665828705, 0.008743247017264366, 0.00883294828236103, 0.010574627667665482, -0.044394854456186295, -0.006912414915859699, 0.012407703325152397, -0.02181057073175907, -0.0030170546378940344, -0.04744565486907959, 0.0248719435185194, -0.0136745385825634, 0.005256884265691042, -0.06572592258453369, -0.04382502660155296, -0.01856866478919983, 0.015017823316156864, -0.0016810684464871883, 0.001966912066563964, 0.06715016067028046, 0.05419166758656502, -0.015016798861324787, 0.008195929229259491, -0.008052538149058819, -0.00010220272815786302, -0.013036645017564297, -0.021501760929822922, 0.0045066759921610355, -0.032204192131757736, 0.02883702702820301, -0.030001165345311165, 0.03489857539534569, -0.04388440027832985, 0.006571061443537474, -0.063425011

In [ ]:
results = com_store.query_vector_index(
    "class_vector_index",
    query_vector=prompt_embedding
)

NameError: name 'pg_store' is not defined

In [ ]:
for result in results:
  print(result["name"])

In [ ]:
#Testing label filters
results = pg_store.query_fulltext_index(
    "module_index",
    query ="bs4 tag",
    labels = ["MODULE"])
for r in results:
  print(r)

## Test querying bs4.Doctype using a sample prompt

In [ ]:
prompt = "allows inspection and manipulation of DOCTYPE declaration"

In [ ]:
prompt_embedding = Settings.embed_model.get_query_embedding(prompt)
print("Length:", len(prompt_embedding))
print("Prompt Embedding:", prompt_embedding)
print()

Length: 1024
Prompt Embedding: [-0.01496047992259264, -0.06415032595396042, 0.04950472712516785, -0.01700117066502571, -0.024948270991444588, -0.009298915974795818, -0.0005513278883881867, -0.0031504782382398844, -0.030203433707356453, 0.04122619330883026, 0.013718781992793083, 0.00837327353656292, 0.006868468597531319, 0.003317068563774228, -0.020653948187828064, 0.011040214449167252, 0.012148918583989143, -0.017772020772099495, -0.08575187623500824, 0.02304573357105255, -0.01623138040304184, 0.028225770220160484, -0.06853199005126953, 0.004591086879372597, -0.04282040521502495, 0.020735125988721848, 0.00897355005145073, -0.010337792336940765, 0.0726991668343544, 0.04805079847574234, -0.02482391707599163, 0.024730928242206573, -0.024091193452477455, -0.012011650949716568, -0.04226236417889595, -0.06704608350992203, 0.02696847915649414, -0.018441449850797653, 0.04458236321806908, -0.018734417855739594, 0.01603034883737564, -0.01616615429520607, -0.007367504294961691, -0.070540323853492

In [ ]:
results = pg_store.query_vector_index(
    "class_vector_index",
    query_vector=prompt_embedding
)

In [ ]:
for result in results:
  print(result["name"])

bs4.Doctype
bs4.Declaration
bs4.ProcessingInstruction
bs4.UnicodeDammit
bs4.CData
bs4.formatter.HTMLFormatter
bs4.Comment
bs4.Stylesheet
bs4.BeautifulSoup
bs4.SoupStrainer


## Test basic code generation code in COMstore for requests.api.get (DO NOT TOUCH)

In [ ]:
entity_name = "requests.get"
entity_name = com_store._get_full_path_name(entity_name, "")
parent_module = com_store._get_parent_module_info(entity_name)
parent_module_repr = com_store._get_retrieved_entity_representation(parent_module[0]) if parent_module else ""

In [ ]:
parent_module_repr

'\n        Name: requests.api\n        Full Path: requests.api\n        Imports: [\'requests.sessions\']\n        Source: ```\n        """\nrequests.api\n~~~~~~~~~~~~\n\nThis module implements the Requests API.\n\n:copyright: (c) 2012 by Kenneth Reitz.\n:license: Apache2, see LICENSE for more details.\n"""\n\nfrom . import sessions\n\n\ndef request(method, url, **kwargs):\n    """Constructs and sends a :class:`Request <Request>`.\n\n    :param method: method for the new :class:`Request` object: ``GET``, ``OPTIONS``, ``HEAD``, ``POST``, ``PUT``, ``PATCH``, or ``DELETE``.\n    :param url: URL for the new :class:`Request` object.\n    :param params: (optional) Dictionary, list of tuples or bytes to send\n        in the query string for the :class:`Request`.\n    :param data: (optional) Dictionary, list of tuples, bytes, or file-like\n        object to send in the body of the :class:`Request`.\n    :param json: (optional) A JSON serializable Python object to send in the body of the :class:

In [ ]:
DEFAULT_DEPENDENCY_PATHS_PROMPT_TEMPLATE.format(
    main_entity_name=entity_name,
    parent_module_information=parent_module_repr
)

'\n    You are a code analysis expert. You are to find all the full paths of parameter types as well as return types for using the following MAIN_ENTITY, given the following information, rules and examples ONLY:\n\n    MAIN_ENTITY name:\n    requests.api.get\n\n    PARENT_MODULE information:\n    \n        Name: requests.api\n        Full Path: requests.api\n        Imports: [\'requests.sessions\']\n        Source: ```\n        """\nrequests.api\n~~~~~~~~~~~~\n\nThis module implements the Requests API.\n\n:copyright: (c) 2012 by Kenneth Reitz.\n:license: Apache2, see LICENSE for more details.\n"""\n\nfrom . import sessions\n\n\ndef request(method, url, **kwargs):\n    """Constructs and sends a :class:`Request <Request>`.\n\n    :param method: method for the new :class:`Request` object: ``GET``, ``OPTIONS``, ``HEAD``, ``POST``, ``PUT``, ``PATCH``, or ``DELETE``.\n    :param url: URL for the new :class:`Request` object.\n    :param params: (optional) Dictionary, list of tuples or bytes t

In [ ]:
await com_store._get_dependencies(entity_name, parent_module_repr)

INFO:root:Dependencies: ['requests.Response']


Dependencies: ['requests.Response']


['requests.Response']

In [ ]:
code = await com_store.generate_code("requests.get")

INFO:root:Dependencies: ['requests.Response']


Dependencies: ['requests.Response']


INFO:root:Initial dependencies for entity requests.api.get: ['requests.Response']


Initial dependencies for entity requests.api.get: ['requests.Response']


INFO:root:Dependencies: ['requests.models.PreparedRequest']


Dependencies: ['requests.models.PreparedRequest']


... (2 outputs left out)

INFO:root:Dependencies: ['requests.models.PreparedRequest']


Dependencies: ['requests.models.PreparedRequest']


INFO:root:Next level dependencies: []


Next level dependencies: []


In [ ]:
print(code)

```python
import requests

def make_get_request(url, params=None, **kwargs):
    """
    Sends a GET request to the specified URL.

    Args:
        url (str): The URL to send the request to.
        params (dict, list of tuples, bytes, optional): Query string parameters. Defaults to None.
        **kwargs: Optional arguments that `requests.get` takes.

    Returns:
        requests.Response: The response object.
    """
    response = requests.get(url, params=params, **kwargs)
    return response

if __name__ == '__main__':
    # Example usage:
    url = "https://httpbin.org/get"
    params = {"key1": "value1", "key2": "value2"}
    
    response = make_get_request(url, params=params)

    print(f"Status Code: {response.status_code}")
    print(f"URL: {response.url}")
    print(f"Headers: {response.headers}")
    print(f"Content: {response.text}")
    
    # Example with no params
    response = make_get_request("https://httpbin.org/ip")
    print(f"Status Code: {response.status_code

In [ ]:
entity_name = "requests.models.Response"
entity_name = com_store._get_full_path_name(entity_name, "")
parent_module = com_store._get_parent_module_info(entity_name)
parent_module_repr = com_store._get_retrieved_entity_representation(parent_module[0]) if parent_module else ""

In [ ]:
parent_module_repr

'\n        Name: requests.models\n        Full Path: requests.models\n        Imports: [\'requests.utils.check_header_validity\', \'requests.hooks.default_hooks\', \'requests.exceptions.MissingSchema\', \'requests.exceptions.StreamConsumedError\', \'requests.cookies.get_cookie_header\', \'requests.structures.CaseInsensitiveDict\', \'requests.utils.requote_uri\', \'requests.utils.get_auth_from_url\', \'requests.exceptions.InvalidJSONError\', \'requests.utils.super_len\', \'requests.exceptions.HTTPError\', \'requests.exceptions.SSLError\', \'requests.exceptions.InvalidURL\', \'requests.utils.stream_decode_response_unicode\', \'requests.utils.parse_header_links\', \'requests.utils.to_key_val_list\', \'requests.exceptions.ConnectionError\', \'requests.exceptions.ContentDecodingError\', \'requests.auth.HTTPBasicAuth\', \'requests.utils.guess_filename\', \'requests.exceptions.ChunkedEncodingError\', \'requests.cookies.cookiejar_from_dict\', \'requests.exceptions.JSONDecodeError\', \'requests

In [ ]:
DEFAULT_DEPENDENCY_PATHS_PROMPT_TEMPLATE.format(
    main_entity_name=entity_name,
    parent_module_information=parent_module_repr
)

'\n    You are a code analysis expert. You are to find all the full paths of parameter types as well as return types for using the following MAIN_ENTITY, given the following information, rules and examples ONLY:\n\n    MAIN_ENTITY name:\n    requests.models.Response\n\n    PARENT_MODULE information:\n    \n        Name: requests.models\n        Full Path: requests.models\n        Imports: [\'requests.utils.check_header_validity\', \'requests.hooks.default_hooks\', \'requests.exceptions.MissingSchema\', \'requests.exceptions.StreamConsumedError\', \'requests.cookies.get_cookie_header\', \'requests.structures.CaseInsensitiveDict\', \'requests.utils.requote_uri\', \'requests.utils.get_auth_from_url\', \'requests.exceptions.InvalidJSONError\', \'requests.utils.super_len\', \'requests.exceptions.HTTPError\', \'requests.exceptions.SSLError\', \'requests.exceptions.InvalidURL\', \'requests.utils.stream_decode_response_unicode\', \'requests.utils.parse_header_links\', \'requests.utils.to_key_v

In [ ]:
await com_store._get_dependencies(entity_name, parent_module_repr)

INFO:root:Dependencies: ['requests.Response']


Dependencies: ['requests.Response']
Dependencies: ['requests.Response']


['requests.Response']

In [ ]:
code = await com_store.generate_code("requests.get")

INFO:root:Dependencies: ['requests.Response']


Dependencies: ['requests.Response']
Dependencies: ['requests.Response']


INFO:root:Initial dependencies for entity requests.api.get: ['requests.Response']


Initial dependencies for entity requests.api.get: ['requests.Response']
Initial dependencies for entity requests.api.get: ['requests.Response']


INFO:root:Dependencies: ['requests.models.Response']


Dependencies: ['requests.models.Response']
Dependencies: ['requests.models.Response']


... (2 outputs left out)

INFO:root:Dependencies: ['requests.models.Response']


Dependencies: ['requests.models.Response']
Dependencies: ['requests.models.Response']


INFO:root:Next level dependencies: []


Next level dependencies: []
Next level dependencies: []


In [ ]:
print(code)

```python
import requests

def get_request(url, params=None, **kwargs):
    """
    Sends a GET request to the specified URL.

    Args:
        url (str): The URL to send the request to.
        params (dict, optional): Query parameters to include in the request. Defaults to None.
        **kwargs: Additional keyword arguments to pass to the requests.get function.

    Returns:
        requests.models.Response: The response object.
    """
    response = requests.get(url, params=params, **kwargs)
    return response

if __name__ == '__main__':
    # Example Usage
    url = "https://httpbin.org/get"  # Replace with the desired URL
    params = {'key1': 'value1', 'key2': 'value2'}
    
    try:
        response = get_request(url, params=params)

        if response.ok:
            print("Request was successful!")
            print("Status Code:", response.status_code)
            print("Response Headers:", response.headers)
            print("Response Content:", response.text)  # Or res

# Test LATS over COM for requests.api.get

In [ ]:
#@title Create necessary types

from llama_index.core import PromptTemplate
from llama_index.core.bridge.pydantic import Field, BaseModel
from llama_index.core.agent.react.types import (
    BaseReasoningStep,
    ResponseReasoningStep,
)
from typing import List, Optional
import math

DEFAULT_COM_REFLECTION_PROMPT_STR = """
Given a query and the current state of graph traversal, evaluate the progress in finding a relevant **METHOD or FUNCTION** within the Python code graph. The primary objective is to find and return the **NAME** of the **METHOD or FUNCTION** that best satisfies the query, not to generate any code.

Consider the following:

- **Correctness**: Are we exploring entities (ROOT_MODULE, MODULE, CLASS, METHOD, FUNCTION) that are potentially relevant to the query, given the graph structure and relationships (ROOT_MODULE-HAS_SUBMODULE->MODULE, MODULE-HAS_SUBMODULE->MODULE, MODULE-HAS_CLASS->CLASS, CLASS-HAS_CLASS->CLASS, CLASS-HAS_METHOD->METHOD, MODULE-HAS_FUNCTION->FUNCTION)?
- **Efficiency**: Are we prioritizing the exploration of entities that have a higher cosine similarity score, indicating a stronger semantic match to the query?
- **Completeness**: Have we found a potential solution (a METHOD or FUNCTION node)? If so, how well does its description match the query based on its cosine similarity score?

Based on your assessment, provide an INTEGER score between 1 and 10, where:
- 10 indicates an almost perfect match: a relevant **METHOD or FUNCTION** has been found, its description strongly aligns with the query, and it has a high cosine similarity score.
- 1 indicates a complete mismatch or a dead end: the current path is unlikely to lead to a relevant **METHOD or FUNCTION**.

Set `is_done` to True **only** when you find a **METHOD or FUNCTION** that satisfies the query. **Do not stop exploring if you find other types of entities.**

Your task is to find and return the **NAME** of the **METHOD or FUNCTION** only. Do not generate any code or explanations.

Query: {query}
Conversation History:
{conversation_history}
"""

DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT_STR = """
Given the current node in the graph and the query, suggest the next steps for traversing the Python code graph to find a relevant **METHOD or FUNCTION**. The primary objective is to find and return the **NAME** of the **METHOD or FUNCTION** that best satisfies the query, not to generate any code.

Consider the following:

- **Node Type and Available Relationships**:
    - If the current node is a METHOD or FUNCTION, suggest evaluating its description against the query to determine if it's a suitable match. **If it's a match, your task is complete. Return the METHOD or FUNCTION's name immediately.**
    - If the current node is a ROOT_MODULE, MODULE, or CLASS, use the `retrieve_relevant_children_in_subtree` tool to find the most relevant immediate children based on cosine similarity to the query. Prioritize exploring children with higher similarity scores.
- **Root Module Exploration**: If the current exploration path seems unpromising, consider using the `retrieve_relevant_root_modules` tool to identify and explore alternative root modules that might contain a relevant METHOD or FUNCTION.
- **Query Relevance**: Prioritize actions that are more likely to lead to a METHOD or FUNCTION closely related to the query.

Suggest **up to {num_candidates}** of the following actions, ensuring that the suggested actions are **distinct and not extremely similar** to each other. It's acceptable to suggest **fewer than {num_candidates}** actions if there are not enough distinct and relevant options:

- "Evaluate the current node" (if it's a METHOD or FUNCTION).
- "Explore the most relevant children of the current node using `retrieve_relevant_children_in_subtree`".

When using the `retrieve_relevant_children_in_subtree` action, make sure your query is descriptive and not just a couple of words.

Remember, your goal is to efficiently find and return the **NAME** of a **METHOD or FUNCTION** whose description closely matches the query. **Stop immediately once a suitable METHOD or FUNCTION is found and return its NAME.**

Query: {query}
Conversation History:
{conversation_history}

Provide your suggested next actions **concisely, focusing on the actions themselves (e.g., "Explore children", "Evaluate node")**.
"""

DEFAULT_COM_REFLECTION_PROMPT = PromptTemplate(DEFAULT_COM_REFLECTION_PROMPT_STR)
DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT = PromptTemplate(DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT_STR)

class COMEvaluation(BaseModel):
    """Evaluation of a given node."""

    score: int = Field(
        description="Score of the reflection indicating **correctness**. Integer from 1-10",
        le=10,
        ge=0,
    )
    is_done: bool = Field(
        False, description="Whether the answer is found yet (**completeness**)."
    )
    reasoning: str = Field(
        default="", description="Reasoning and justification for the evaluation."
    )
    entity_name: str = Field(
        default="", description="Name of the entity IF the answer is found."
    )

class COMSearchNode(BaseModel):
    """Search node.

    Named differently from `Node` which is a core module in LlamaIndex.

    """

    current_reasoning: List[BaseReasoningStep] = Field(
        ..., description="Current reasoning."
    )
    parent: Optional["COMSearchNode"] = Field(default=None, description="Parent node.")
    children: List["COMSearchNode"] = Field(
        default_factory=list, description="Children nodes."
    )
    evaluation: COMEvaluation = Field(..., description="Evaluation of the node.")
    visits: int = Field(default=0, description="Number of visits to the node.")

    @property
    def answer(self) -> Optional[str]:
        """Answer."""
        if not self.current_reasoning:
            return None

        if isinstance(self.current_reasoning[-1], ResponseReasoningStep):
            return self.current_reasoning[-1].response
        else:
            return self.current_reasoning[-1].get_content()

    @property
    def is_done(self) -> bool:
        """Is the node done."""
        return self.evaluation.is_done

    @property
    def score(self) -> float:
        """Score of the node."""
        return self.evaluation.score

    @property
    def upper_confidence_bound(self) -> float:
        """Upper confidence bound."""
        return self.score + 1.0 * math.sqrt(math.log(self.parent.visits) / self.visits)

    def backpropagate(self, reward: float) -> None:
        """Backpropagate the reward."""
        cur_node = self
        while cur_node is not None:
            cur_node.visits += 1
            cur_node.evaluation.score = (
                reward + (cur_node.visits - 1) * cur_node.score
            ) / cur_node.visits
            cur_node = cur_node.parent

    def get_best_leaf(self) -> "COMSearchNode":
        """Get best leaf node.

        Get best leaf node across any children nodes.

        """
        # only get children that aren't done yet
        free_children = [c for c in self.children if not c.is_done]
        if not free_children:
            return self

        best_child = max(free_children, key=lambda x: x.upper_confidence_bound)
        return best_child.get_best_leaf()

In [ ]:
#@title Create COMSearchAgentWorker

from llama_index.agent.lats import LATSAgentWorker
from llama_index.core.agent import (
    AgentChatResponse,
    ReActOutputParser,
    Task,
)
# from llama_index.core.prompts import PromptTemplate
from typing import Any, Dict, List, Optional, Tuple, cast
from llama_index.core.utils import print_text
from llama_index.core.agent.react.types import (
    ActionReasoningStep,
    BaseReasoningStep,
    ObservationReasoningStep,
    ResponseReasoningStep,
)
# from llama_index.core.bridge.pydantic import Field
from llama_index.agent.lats.types import Candidates
from llama_index.core.tools import ToolSelection, acall_tool_with_selection
# from .types import COMEvaluation, COMSearchNode, DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT, DEFAULT_COM_REFLECTION_PROMPT
import asyncio

class COMSearchAgentWorker(LATSAgentWorker):
    """COM Search Agent Worker class."""
    com_search_agent: Any = None

    def __init__(self, *args, reflection_prompt=None, candidate_expansion_prompt=None, **kwargs):
        # candidate_expansion_prompt = candidate_expansion_prompt or DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT
        super().__init__(
            *args,
            reflection_prompt=reflection_prompt or DEFAULT_COM_REFLECTION_PROMPT,
            candiate_expansion_prompt=candidate_expansion_prompt or DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT,
            **kwargs,
        )

    def _initialize_state(self, task: Task, **kwargs: Any) -> Dict[str, Any]:
        """Initialize state."""
        # initialize root node
        root_node = COMSearchNode(
            current_reasoning=[ObservationReasoningStep(observation=task.input)],
            evaluation=COMEvaluation(score=1),  # evaluation for root node is blank
        )
        return {"count": 0, "solution_queue": [], "root_node": root_node}

    async def _arun_candidate(
        self,
        node: COMSearchNode,
        task: Task,
    ) -> List[BaseReasoningStep]:
        """Generate candidate for a given node.

        Generically we sample the action space to generate new candidate nodes.

        Practically since we're using a ReAct powered agent, this means
        using generating a ReAct trajectory, running a tool.

        """
        output_parser = ReActOutputParser()
        # format react prompt
        formatted_prompt = self.chat_formatter.format(
            self.tools,
            chat_history=task.memory.get(),
            current_reasoning=node.current_reasoning,
        )
        # run LLM
        response = await self.llm.achat(formatted_prompt)
        # parse output into reasoning step

        try:
            reasoning_step = output_parser.parse(response.message.content)
        except ValueError as e:
            reasoning_step = ResponseReasoningStep(
                thought=response.message.content,
                response=f"Encountered an error parsing: {e!s}",
            )
        # get response or run tool
        if reasoning_step.is_done:
            reasoning_step = cast(ResponseReasoningStep, reasoning_step)
            current_reasoning = [reasoning_step]
        else:
            reasoning_step = cast(ActionReasoningStep, reasoning_step)
            tool_selection = ToolSelection(
                tool_id=reasoning_step.action,
                tool_name=reasoning_step.action,
                tool_kwargs=reasoning_step.action_input,
            )
            try:
                tool_output = await acall_tool_with_selection(
                    tool_selection, self.tools, verbose=self.verbose
                )
            except Exception as e:
                tool_output = f"Encountered error: {e!s}"
            observation_step = ObservationReasoningStep(observation=str(tool_output))
            current_reasoning = [reasoning_step, observation_step]

        return current_reasoning

    async def _get_next_candidates(
        self,
        cur_node: COMSearchNode,
        input: str,
    ) -> List[str]:
        """Get next candidates."""
        # get candidates
        history_str = "\n".join([s.get_content() for s in cur_node.current_reasoning])

        candidates = await self.llm.astructured_predict(
            Candidates,
            prompt=self.candiate_expansion_prompt,
            query=input,
            conversation_history=history_str,
            num_candidates=self.num_expansions,
        )
        candidate_strs = candidates.candidates[: self.num_expansions]
        if self.verbose:
            print_text(f"> Got candidates: {candidate_strs}\n", color="yellow")

        return candidate_strs

    def _initialize_state(self, task: Task, **kwargs: Any) -> Dict[str, Any]:
        """Initialize state."""
        # initialize root node
        root_node = COMSearchNode(
            current_reasoning=[ObservationReasoningStep(observation=task.input)],
            evaluation=COMEvaluation(score=1),  # evaluation for root node is blank
        )
        return {"count": 0, "solution_queue": [], "root_node": root_node}

    async def _aevaluate(
        self,
        cur_node: COMSearchNode,
        current_reasoning: List[BaseReasoningStep],
        input: str,
    ) -> float:
        """Evaluate."""
        all_reasoning = cur_node.current_reasoning + current_reasoning
        history_str = "\n".join([s.get_content() for s in all_reasoning])
        evaluation = await self.llm.astructured_predict(
            COMEvaluation,
            prompt=self.reflection_prompt,
            query=input,
            conversation_history=history_str,
        )
        if self.verbose:
            print_text(
                f"> Evaluation for input {input}\n: {evaluation}\n\n", color="pink"
            )

        return evaluation

    def _update_state(
        self,
        node: COMSearchNode,
        current_reasoning: List[BaseReasoningStep],
        evaluation: COMEvaluation,
    ) -> COMSearchNode:
        """Update state."""
        # create child node
        new_node = COMSearchNode(
            current_reasoning=node.current_reasoning + current_reasoning,
            parent=node,
            children=[],
            evaluation=evaluation,
        )
        node.children.append(new_node)

        # backpropagate the reward
        new_node.backpropagate(evaluation.score)

        return new_node

    async def _arun_step(
        self, state: Dict[str, Any], task: Task, input: Optional[str] = None
    ) -> Tuple[AgentChatResponse, bool]:
        """Run step.

        Returns:
            Tuple of (agent_response, is_done)

        """
        root_node = state["root_node"]
        cur_node = root_node.get_best_leaf()
        if self.verbose:
            print_text(
                f"> Selecting node to expand: {cur_node.answer}\n", color="green"
            )

        # expand the given node, generate n candidate nodes
        new_candidates = await self._get_next_candidates(
            cur_node,
            task.input,
        )

        new_nodes = []
        for candidate in new_candidates:
            new_nodes.append(
                self._update_state(
                    cur_node,
                    [ObservationReasoningStep(observation=candidate)],
                    COMEvaluation(score=1),  # evaluation for candidate node is blank
                )
            )

        # expand the given node, generate n candidates
        # for each candidate, run tool, get response

        solution_queue: List[COMSearchNode] = state["solution_queue"]

        # first, generate the candidates
        candidate_jobs = [
            self._arun_candidate(new_node, task) for new_node in new_nodes
        ]
        all_new_reasoning_steps = await asyncio.gather(*candidate_jobs)
        if self.verbose:
            for new_reasoning_steps in all_new_reasoning_steps:
                out_txt = "\n".join([s.get_content() for s in new_reasoning_steps])
                print_text(f"> Generated new reasoning step: {out_txt}\n", color="blue")
        # then, evaluate the candidates
        eval_jobs = [
            self._aevaluate(new_node, new_reasoning_steps, task.input)
            for new_node, new_reasoning_steps in zip(new_nodes, all_new_reasoning_steps)
        ]
        evaluations = await asyncio.gather(*eval_jobs)
        # then, update the state
        for new_reasoning_steps, cur_new_node, evaluation in zip(
            all_new_reasoning_steps, new_nodes, evaluations
        ):
            new_node = self._update_state(cur_new_node, new_reasoning_steps, evaluation)
            if new_node.is_done:
                if self.verbose:
                    print_text(
                        f"> Found solution node: {new_node.answer}\n", color="cyan"
                    )
                solution_queue.append(new_node)

        # check if done
        state["count"] += 1
        if solution_queue:
            is_done = True
        elif self.max_rollouts > 0 and state["count"] >= self.max_rollouts:
            is_done = True
        else:
            is_done = False

        # determine response
        if solution_queue:
            best_solution_node = max(solution_queue, key=lambda x: x.score)
            response = best_solution_node.evaluation.entity_name
        else:
            response = "I am still thinking."

        if self.verbose:
            print_text(f"> Got final response: {response!s}\n", color="green")

        # return response
        return AgentChatResponse(response=str(response)), is_done

    async def retrieve_relevant_entity(self, input: str) -> str:
        """
        Retrieve relevant entity given an input query.

        Args:
            input (str): Input to the agent.

        Returns:
            str: Relevant entity name.
        """
        # Initialize agent if not already initialized
        if self.com_search_agent is None:
            self.com_search_agent = self.as_agent()

        # Run agent
        response = await self.com_search_agent.achat(input)

        # Reset agent state
        self.com_search_agent.reset()

        return response.response

In [ ]:
#@title Create necessary tools

from llama_index.core.tools import FunctionTool, ToolMetadata

retrieve_root_modules_tool = FunctionTool.from_defaults(
    fn=com_store.retrieve_relevant_root_modules,
    description="Retrieve relevant root modules based on the query, using cosine similarity search"
)
retrieve_children_tool = FunctionTool.from_defaults(
    fn=com_store.retrieve_relevant_children_in_subtree,
    description="Retrieve relevant immediate children of the current node based on the query, using cosine similarity search"
)

In [ ]:
#@title Initialize the COMSearchAgentWorker

agent_worker = COMSearchAgentWorker.from_tools(
    tools=[
        retrieve_root_modules_tool,
        retrieve_children_tool,
    ],
    llm=Settings.llm,
    verbose=True,
    num_expansions=2,
    max_rollouts=3,
)

agent = agent_worker.as_agent()

## Following is a working example (DO NOT TOUCH)

In [ ]:
response = await agent.achat("Find a functionality that allows retrieving data from a URL")

> Selecting node to expand: Observation: Find a functionality that allows retrieving data from a URL
> Got candidates: ['Explore the most relevant children of the current node using `retrieve_relevant_children_in_subtree`']
=== Calling Function ===
Calling function: retrieve_relevant_root_modules with args: {"query": "retrieve data from URL", "top_k": 5}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
[{'name': 'urllib', 'summary': '### Summary of `urllib` Module\n\n#### Overview\nThe `urllib` module in Python is a package that collects several modules for working with URLs. It provides tools for opening URLs using various protocols and includes functionalities for parsing and manipulating URLs.\n\n#### Sub-Modules\n\n1. **urllib.request**\n   - Provides classes and functions for opening and reading URLs. It handles HTTP, FTP, and other protocols.\n   - Includes classes like `Request`, `OpenerDirector`, and `HTTPHandler` for customizing and managing requests.\n   - Supports automatic redirection, handling of HTTP errors, and proxy support.\n\n2. **urllib.response**\n   - Provides classes for handling HTTP responses in a file-like manner.\n   - Includes base classes like `addbase`, `addclosehook`, `addinfo`, and `addinfourl` for different functionalities.\n   - Key classes like `addinfourl` provide methods for retrieving the URL, status code, and headers.\n\n3

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
[]
=== Calling Function ===
Calling function: retrieve_relevant_children_in_subtree with args: {"query": "retrieve data from URL", "parent_name": "urllib"}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
[]
> Generated new reasoning step: Thought: The `urllib` and `requests` modules seem most relevant for retrieving data from a URL. I will start by exploring the children of the `urllib` module to see if I can find more specific functionalities.
Action: retrieve_relevant_children_in_subtree
Action Input: {'query': 'retrieve data from URL', 'parent_name': 'urllib'}
Observation: []
> Generated new reasoning step: Thought: The `urllib` and `requests` modules seem most relevant for retrieving data from a URL. I will start by exploring the children of the `urllib` module to see if I can find more specific functionalities.
Action: retrieve_relevant_children_in_subtree
Action Input: {'query': 'retrieve data from URL', 'parent_name': 'urllib'}
Observation: []
> Evaluation for input Find a functionality that allows retrieving data from a URL
: score=6 is_done=False reasoning="The `urllib` module is relevant, but exploring its children didn't yield any immediate results. T

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
[{'name': 'requests.check_compatibility', 'score': 0.7694265842437744, 'labels': ['FUNCTION'], 'summary': 'The `check_compatibility` function is responsible for verifying the compatibility of the `requests` library with its dependencies, specifically `urllib3`, `chardet`, and `charset_normalizer`. It ensures that the installed versions of these libraries meet the minimum requirements for `requests` to function correctly.\n\nThe function first parses the version string of `urllib3`, handling cases where the version might be abbreviated (e.g., "1.16"). It then asserts that `urllib3` is at least version 1.21.1. This check ensures that `requests` is using a compatible version of the underlying HTTP library.\n\nNext, the function checks for the presence and version of either `chardet` or `charset_normalizer`, which are used for character encoding detection. If `chardet` is present, it verifies that its version is between 3.0.2 (inclusive) and 6.0.0 (exclusive). If `c

In [ ]:
agent.reset()

In [ ]:
response

AgentChatResponse(response='requests.api.get', sources=[], source_nodes=[], is_dummy_stream=False, metadata=None)

## Following is a working example (DO NOT TOUCH)

In [ ]:
response = await agent.achat("Find a functionality that allows retrieving data from a URL")

> Selecting node to expand: Observation: Find a functionality that allows retrieving data from a URL
> Got candidates: ['Explore the most relevant children of the current node using `retrieve_relevant_children_in_subtree`']
=== Calling Function ===
Calling function: retrieve_relevant_root_modules with args: {"query": "retrieve data from URL", "top_k": 5}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Root Modules:

1. **urllib**: ### Summary of `urllib` Module

#### Overview
The `urllib` module in Python is a package that collects several modules for working with URLs. It provides tools for opening URLs using various protocols and includes functionalities for parsing and manipulating URLs.

#### Sub-Modules

1. **urllib.request**
   - Provides classes and functions for opening and reading URLs. It handles HTTP, FTP, and other protocols.
   - Includes classes like `Request`, `OpenerDirector`, and `HTTPHandler` for customizing and managing requests.
   - Supports automatic redirection, handling of HTTP errors, and proxy support.

2. **urllib.response**
   - Provides classes for handling HTTP responses in a file-like manner.
   - Includes base classes like `addbase`, `addclosehook`, `addinfo`, and `addinfourl` for different functionalities.
   - Key classes like `addinfourl` provide methods for retrieving the URL, status code, and headers.

3. **urllib

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Children in Subtree:

1. **urllib.request**: ### Summary of `urllib.request`

#### Member Summaries:
- **Project Description**: The `urllib.request` module defines functions and classes which help in opening URLs (mostly HTTP) in a complex world — basic and digest authentication, redirections, cookies, and more.
- **PyPi Description**: Not provided in the source code.

#### Analysis:
The `urllib.request` module in Python provides a suite of classes and functions for opening and reading URLs. It handles various aspects of HTTP requests, including authentication, redirections, and cookies. The module is designed to be flexible and robust, supporting a wide range of protocols and features.

#### Key Functionalities:
1. **Opening URLs**: The core functionality is opening URLs using a variety of protocols, with support for HTTP, HTTPS, FTP, and more.
2. **Authentication**: Handles both basic and digest authentication, allowing users to authenticate with serv

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Children in Subtree:

1. **requests.check_compatibility**: The `check_compatibility` function is responsible for verifying the compatibility of the `requests` library with its dependencies, specifically `urllib3`, `chardet`, and `charset_normalizer`. It ensures that the installed versions of these libraries meet the minimum requirements for `requests` to function correctly.

The function first parses the version string of `urllib3`, handling cases where the version might be abbreviated (e.g., "1.16"). It then asserts that `urllib3` is at least version 1.21.1. This check ensures that `requests` is using a compatible version of the underlying HTTP library.

Next, the function checks for the presence and version of either `chardet` or `charset_normalizer`, which are used for character encoding detection. If `chardet` is present, it verifies that its version is between 3.0.2 (inclusive) and 6.0.0 (exclusive). If `chardet` is not found, it checks for `charse

In [ ]:
agent.reset()

In [ ]:
response

AgentChatResponse(response='requests.api.get', sources=[], source_nodes=[], is_dummy_stream=False, metadata=None)

# Simple Multi Agent Workflow

In [ ]:
#@title Create necessary types
from llama_index.core import PromptTemplate
from llama_index.core.bridge.pydantic import Field, BaseModel
from llama_index.core.agent.react.types import (
    BaseReasoningStep,
    ResponseReasoningStep,
)
from typing import List, Optional
import math

DEFAULT_COM_REFLECTION_PROMPT_STR = """
Given a query and the current state of graph traversal, evaluate the progress in finding a relevant **METHOD or FUNCTION** within the Python code graph. The primary objective is to find and return the **NAME** of the **METHOD or FUNCTION** that best satisfies the query, not to generate any code.

Consider the following:

- **Correctness**: Are we exploring entities (ROOT_MODULE, MODULE, CLASS, METHOD, FUNCTION) that are potentially relevant to the query, given the graph structure and relationships (ROOT_MODULE-HAS_SUBMODULE->MODULE, MODULE-HAS_SUBMODULE->MODULE, MODULE-HAS_CLASS->CLASS, CLASS-HAS_CLASS->CLASS, CLASS-HAS_METHOD->METHOD, MODULE-HAS_FUNCTION->FUNCTION)?
- **Efficiency**: Are we prioritizing the exploration of entities that have a higher cosine similarity score, indicating a stronger semantic match to the query?
- **Completeness**: Have we found a potential solution (a METHOD or FUNCTION node)? If so, how well does its description match the query based on its cosine similarity score?

Based on your assessment, provide an INTEGER score between 1 and 10, where:
- 10 indicates an almost perfect match: a relevant **METHOD or FUNCTION** has been found, its description strongly aligns with the query, and it has a high cosine similarity score.
- 1 indicates a complete mismatch or a dead end: the current path is unlikely to lead to a relevant **METHOD or FUNCTION**.

Set `is_done` to True **only** when you find a **METHOD or FUNCTION** that satisfies the query. **Do not stop exploring if you find other types of entities.**

Your task is to find and return the **NAME** of the **METHOD or FUNCTION** only. Do not generate any code or explanations.

Query: {query}
Conversation History:
{conversation_history}
"""

DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT_STR = """
Given the current node in the graph and the query, suggest the next steps for traversing the Python code graph to find a relevant **METHOD or FUNCTION**. The primary objective is to find and return the **NAME** of the **METHOD or FUNCTION** that best satisfies the query, not to generate any code.

Consider the following:

- **Node Type and Available Relationships**:
    - If the current node is a METHOD or FUNCTION, suggest evaluating its description against the query to determine if it's a suitable match. **If it's a match, your task is complete. Return the METHOD or FUNCTION's name immediately.**
    - If the current node is a ROOT_MODULE, MODULE, or CLASS, use the `retrieve_relevant_children_in_subtree` tool to find the most relevant immediate children based on cosine similarity to the query. Prioritize exploring children with higher similarity scores.
- **Root Module Exploration**: If the current exploration path seems unpromising, consider using the `retrieve_relevant_root_modules` tool to identify and explore alternative root modules that might contain a relevant METHOD or FUNCTION.
- **Query Relevance**: Prioritize actions that are more likely to lead to a METHOD or FUNCTION closely related to the query.

Suggest **up to {num_candidates}** of the following actions, ensuring that the suggested actions are **distinct and not extremely similar** to each other. It's acceptable to suggest **fewer than {num_candidates}** actions if there are not enough distinct and relevant options:

- "Evaluate the current node" (if it's a METHOD or FUNCTION).
- "Explore the most relevant children of the current node using `retrieve_relevant_children_in_subtree`".

When using the `retrieve_relevant_children_in_subtree` action, make sure your query is descriptive and not just a couple of words.

Remember, your goal is to efficiently find and return the **NAME** of a **METHOD or FUNCTION** whose description closely matches the query. **Stop immediately once a suitable METHOD or FUNCTION is found and return its NAME.**

Query: {query}
Conversation History:
{conversation_history}

Provide your suggested next actions **concisely, focusing on the actions themselves (e.g., "Explore children", "Evaluate node")**.
"""

DEFAULT_COM_REFLECTION_PROMPT = PromptTemplate(DEFAULT_COM_REFLECTION_PROMPT_STR)
DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT = PromptTemplate(DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT_STR)

class COMEvaluation(BaseModel):
    """Evaluation of a given node."""

    score: int = Field(
        description="Score of the reflection indicating **correctness**. Integer from 1-10",
        le=10,
        ge=0,
    )
    is_done: bool = Field(
        False, description="Whether the answer is found yet (**completeness**)."
    )
    reasoning: str = Field(
        default="", description="Reasoning and justification for the evaluation."
    )
    entity_name: str = Field(
        default="", description="Name of the entity IF the answer is found."
    )

class COMSearchNode(BaseModel):
    """Search node.

    Named differently from `Node` which is a core module in LlamaIndex.

    """

    current_reasoning: List[BaseReasoningStep] = Field(
        ..., description="Current reasoning."
    )
    parent: Optional["COMSearchNode"] = Field(default=None, description="Parent node.")
    children: List["COMSearchNode"] = Field(
        default_factory=list, description="Children nodes."
    )
    evaluation: COMEvaluation = Field(..., description="Evaluation of the node.")
    visits: int = Field(default=0, description="Number of visits to the node.")

    @property
    def answer(self) -> Optional[str]:
        """Answer."""
        if not self.current_reasoning:
            return None

        if isinstance(self.current_reasoning[-1], ResponseReasoningStep):
            return self.current_reasoning[-1].response
        else:
            return self.current_reasoning[-1].get_content()

    @property
    def is_done(self) -> bool:
        """Is the node done."""
        return self.evaluation.is_done

    @property
    def score(self) -> float:
        """Score of the node."""
        return self.evaluation.score

    @property
    def upper_confidence_bound(self) -> float:
        """Upper confidence bound."""
        return self.score + 1.0 * math.sqrt(math.log(self.parent.visits) / self.visits)

    def backpropagate(self, reward: float) -> None:
        """Backpropagate the reward."""
        cur_node = self
        while cur_node is not None:
            cur_node.visits += 1
            cur_node.evaluation.score = (
                reward + (cur_node.visits - 1) * cur_node.score
            ) / cur_node.visits
            cur_node = cur_node.parent

    def get_best_leaf(self) -> "COMSearchNode":
        """Get best leaf node.

        Get best leaf node across any children nodes.

        """
        # only get children that aren't done yet
        free_children = [c for c in self.children if not c.is_done]
        if not free_children:
            return self

        best_child = max(free_children, key=lambda x: x.upper_confidence_bound)
        return best_child.get_best_leaf()

In [ ]:
#@title Create COMSearchAgentWorker

from llama_index.agent.lats import LATSAgentWorker
from llama_index.core.agent import (
    AgentChatResponse,
    ReActOutputParser,
    Task,
)
# from llama_index.core.prompts import PromptTemplate
from typing import Any, Dict, List, Optional, Tuple, cast
from llama_index.core.utils import print_text
from llama_index.core.agent.react.types import (
    ActionReasoningStep,
    BaseReasoningStep,
    ObservationReasoningStep,
    ResponseReasoningStep,
)
# from llama_index.core.bridge.pydantic import Field
from llama_index.agent.lats.types import Candidates
from llama_index.core.tools import ToolSelection, acall_tool_with_selection
# from .types import COMEvaluation, COMSearchNode, DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT, DEFAULT_COM_REFLECTION_PROMPT
import asyncio

class COMSearchAgentWorker(LATSAgentWorker):
    """COM Search Agent Worker class."""
    com_search_agent: Any = None

    def __init__(self, *args, reflection_prompt=None, candidate_expansion_prompt=None, **kwargs):
        # candidate_expansion_prompt = candidate_expansion_prompt or DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT
        super().__init__(
            *args,
            reflection_prompt=reflection_prompt or DEFAULT_COM_REFLECTION_PROMPT,
            candiate_expansion_prompt=candidate_expansion_prompt or DEFAULT_COM_CANDIDATE_EXPANSION_PROMPT,
            **kwargs,
        )

    def _initialize_state(self, task: Task, **kwargs: Any) -> Dict[str, Any]:
        """Initialize state."""
        # initialize root node
        root_node = COMSearchNode(
            current_reasoning=[ObservationReasoningStep(observation=task.input)],
            evaluation=COMEvaluation(score=1),  # evaluation for root node is blank
        )
        return {"count": 0, "solution_queue": [], "root_node": root_node}

    async def _arun_candidate(
        self,
        node: COMSearchNode,
        task: Task,
    ) -> List[BaseReasoningStep]:
        """Generate candidate for a given node.

        Generically we sample the action space to generate new candidate nodes.

        Practically since we're using a ReAct powered agent, this means
        using generating a ReAct trajectory, running a tool.

        """
        output_parser = ReActOutputParser()
        # format react prompt
        formatted_prompt = self.chat_formatter.format(
            self.tools,
            chat_history=task.memory.get(),
            current_reasoning=node.current_reasoning,
        )
        # run LLM
        response = await self.llm.achat(formatted_prompt)
        # parse output into reasoning step

        try:
            reasoning_step = output_parser.parse(response.message.content)
        except ValueError as e:
            reasoning_step = ResponseReasoningStep(
                thought=response.message.content,
                response=f"Encountered an error parsing: {e!s}",
            )
        # get response or run tool
        if reasoning_step.is_done:
            reasoning_step = cast(ResponseReasoningStep, reasoning_step)
            current_reasoning = [reasoning_step]
        else:
            reasoning_step = cast(ActionReasoningStep, reasoning_step)
            tool_selection = ToolSelection(
                tool_id=reasoning_step.action,
                tool_name=reasoning_step.action,
                tool_kwargs=reasoning_step.action_input,
            )
            try:
                tool_output = await acall_tool_with_selection(
                    tool_selection, self.tools, verbose=self.verbose
                )
            except Exception as e:
                tool_output = f"Encountered error: {e!s}"
            observation_step = ObservationReasoningStep(observation=str(tool_output))
            current_reasoning = [reasoning_step, observation_step]

        return current_reasoning

    async def _get_next_candidates(
        self,
        cur_node: COMSearchNode,
        input: str,
    ) -> List[str]:
        """Get next candidates."""
        # get candidates
        history_str = "\n".join([s.get_content() for s in cur_node.current_reasoning])

        candidates = await self.llm.astructured_predict(
            Candidates,
            prompt=self.candiate_expansion_prompt,
            query=input,
            conversation_history=history_str,
            num_candidates=self.num_expansions,
        )
        candidate_strs = candidates.candidates[: self.num_expansions]
        if self.verbose:
            print_text(f"> Got candidates: {candidate_strs}\n", color="yellow")

        return candidate_strs

    def _initialize_state(self, task: Task, **kwargs: Any) -> Dict[str, Any]:
        """Initialize state."""
        # initialize root node
        root_node = COMSearchNode(
            current_reasoning=[ObservationReasoningStep(observation=task.input)],
            evaluation=COMEvaluation(score=1),  # evaluation for root node is blank
        )
        return {"count": 0, "solution_queue": [], "root_node": root_node}

    async def _aevaluate(
        self,
        cur_node: COMSearchNode,
        current_reasoning: List[BaseReasoningStep],
        input: str,
    ) -> float:
        """Evaluate."""
        all_reasoning = cur_node.current_reasoning + current_reasoning
        history_str = "\n".join([s.get_content() for s in all_reasoning])
        evaluation = await self.llm.astructured_predict(
            COMEvaluation,
            prompt=self.reflection_prompt,
            query=input,
            conversation_history=history_str,
        )
        if self.verbose:
            print_text(
                f"> Evaluation for input {input}\n: {evaluation}\n\n", color="pink"
            )

        return evaluation

    def _update_state(
        self,
        node: COMSearchNode,
        current_reasoning: List[BaseReasoningStep],
        evaluation: COMEvaluation,
    ) -> COMSearchNode:
        """Update state."""
        # create child node
        new_node = COMSearchNode(
            current_reasoning=node.current_reasoning + current_reasoning,
            parent=node,
            children=[],
            evaluation=evaluation,
        )
        node.children.append(new_node)

        # backpropagate the reward
        new_node.backpropagate(evaluation.score)

        return new_node

    async def _arun_step(
        self, state: Dict[str, Any], task: Task, input: Optional[str] = None
    ) -> Tuple[AgentChatResponse, bool]:
        """Run step.

        Returns:
            Tuple of (agent_response, is_done)

        """
        root_node = state["root_node"]
        cur_node = root_node.get_best_leaf()
        if self.verbose:
            print_text(
                f"> Selecting node to expand: {cur_node.answer}\n", color="green"
            )

        # expand the given node, generate n candidate nodes
        new_candidates = await self._get_next_candidates(
            cur_node,
            task.input,
        )

        new_nodes = []
        for candidate in new_candidates:
            new_nodes.append(
                self._update_state(
                    cur_node,
                    [ObservationReasoningStep(observation=candidate)],
                    COMEvaluation(score=1),  # evaluation for candidate node is blank
                )
            )

        # expand the given node, generate n candidates
        # for each candidate, run tool, get response

        solution_queue: List[COMSearchNode] = state["solution_queue"]

        # first, generate the candidates
        candidate_jobs = [
            self._arun_candidate(new_node, task) for new_node in new_nodes
        ]
        all_new_reasoning_steps = await asyncio.gather(*candidate_jobs)
        if self.verbose:
            for new_reasoning_steps in all_new_reasoning_steps:
                out_txt = "\n".join([s.get_content() for s in new_reasoning_steps])
                print_text(f"> Generated new reasoning step: {out_txt}\n", color="blue")
        # then, evaluate the candidates
        eval_jobs = [
            self._aevaluate(new_node, new_reasoning_steps, task.input)
            for new_node, new_reasoning_steps in zip(new_nodes, all_new_reasoning_steps)
        ]
        evaluations = await asyncio.gather(*eval_jobs)
        # then, update the state
        for new_reasoning_steps, cur_new_node, evaluation in zip(
            all_new_reasoning_steps, new_nodes, evaluations
        ):
            new_node = self._update_state(cur_new_node, new_reasoning_steps, evaluation)
            if new_node.is_done:
                if self.verbose:
                    print_text(
                        f"> Found solution node: {new_node.answer}\n", color="cyan"
                    )
                solution_queue.append(new_node)

        # check if done
        state["count"] += 1
        if solution_queue:
            is_done = True
        elif self.max_rollouts > 0 and state["count"] >= self.max_rollouts:
            is_done = True
        else:
            is_done = False

        # determine response
        if solution_queue:
            best_solution_node = max(solution_queue, key=lambda x: x.score)
            response = best_solution_node.evaluation.entity_name
        else:
            response = "I am still thinking."

        if self.verbose:
            print_text(f"> Got final response: {response!s}\n", color="green")

        # return response
        return AgentChatResponse(response=str(response)), is_done

    async def retrieve_relevant_entity(self, input: str) -> str:
        """
        Retrieve relevant entity given an input query.

        Args:
            input (str): Input to the agent.

        Returns:
            str: Relevant entity name.
        """
        # Initialize agent if not already initialized
        if self.com_search_agent is None:
            self.com_search_agent = self.as_agent()

        # Run agent
        response = await self.com_search_agent.achat(input)

        # Reset agent state
        self.com_search_agent.reset()

        return response.response

In [ ]:
#@title Initialize the COMSearchAgentWorker

from llama_index.core.tools import FunctionTool, ToolMetadata

retrieve_root_modules_tool = FunctionTool.from_defaults(
    fn=com_store.retrieve_relevant_root_modules,
    description="Retrieve relevant root modules based on the query, using cosine similarity search"
)
retrieve_children_tool = FunctionTool.from_defaults(
    fn=com_store.retrieve_relevant_children_in_subtree,
    description="Retrieve relevant immediate children of the current node based on the query, using cosine similarity search"
)

agent_worker = COMSearchAgentWorker.from_tools(
    tools=[
        retrieve_root_modules_tool,
        retrieve_children_tool,
    ],
    llm=Settings.llm,
    verbose=True,
    num_expansions=2,
    max_rollouts=3,
)

In [ ]:
#@title Create necessary tools

from llama_index.tools.duckduckgo import DuckDuckGoSearchToolSpec

ddgs_tools = DuckDuckGoSearchToolSpec().to_tool_list()
code_gen_tool = FunctionTool.from_defaults(
    fn=com_store.generate_code,
    description="Generate code for an entity, with instructions if necessary"
)
retrieve_relevant_entity = FunctionTool.from_defaults(
    fn=agent_worker.retrieve_relevant_entity,
    description="Retrieve best entity satisfying a functionality by searching the code related knowledge graph"
)

all_tools = ddgs_tools + [retrieve_root_modules_tool, code_gen_tool, retrieve_relevant_entity]

In [ ]:
#@title Create Agent Workflow

system_prompt = """
You are a powerful function agent designed to answer coding-related questions by either generating or extracting existing Python code implementations. Always prioritize using existing implementations whenever possible.

You have access to the following tools:

1. retrieve_root_modules_tool (description: Use this to retrieve relevant root modules from the code knowledge graph. *Always start with this tool* to understand the available code, UNLESS the user explicitly asks to search for *new* or unavailable modules.)
2. retrieve_relevant_entity (description: Retrieve the *best* single entity (method, function, class) from the code knowledge graph that satisfies a focused sub-query. Phrase the sub-query as "Find a functionality that..." and specify a relevant root module to limit the search.)
3. code_gen_tool (description: Generate code for a *specific* entity or to combine elements. Use this sparingly, primarily to augment existing code or create small snippets that are essential for connecting retrieved entities.)
4. DuckDuckGo Search Tool (description: Use this *ONLY* if you are *explicitly asked* to search for modules or functionalities *not found* in the code knowledge graph. This should be a last resort.)

Here's how to approach each query:

**1. Initial Assessment (Unless explicitly asked to search for new modules):**

*   *Always* begin by using `retrieve_root_modules_tool` with the user's query. This will give you an overview of the relevant code components in the knowledge graph.

**2. Handling Different Query Types:**

*   **If the query is COMPLEX (requiring multiple components, planning, and graph traversal):**
    1. After using `retrieve_root_modules_tool`, dissect the user's query into simpler, focused sub-queries.
    2. For each sub-query:
        * Use `retrieve_relevant_entity` to find an existing code component (method, function, class) within one of the root modules identified earlier, focusing on the best, single function per step. You may mention A MAXIMUM OF 1 MODULE, if needed.
        * Follow each call to `retrieve_relevant_entity` with `code_gen_tool`, to add instructions for that and generate the needed code. This way you want to use what the function can do, as well as specify, based on any known modules, to create the most accurate and useful code.
*   **If the query requires using new functions that are not found in the knowledge graph:**
     1. use DuckDuckGo and search the web to retrieve any relevant resources and modules.
     2. return them to the user as suggestions

Key Points:

* Do not begin any process by using the DDG unless it is absolutely required.
* Use the retrieved modules to first understand if the query can be full-filled with all that is on that module and the relationships. Only then should you generate a plan.*
* If there are a small number of functions, do not try to create atomic tasks to do so.*
"""

from llama_index.core.agent.workflow import AgentWorkflow

agent_workflow = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=all_tools,
    system_prompt=system_prompt,
)

## Following is a working example of a basic agent workflow running for requests.api.get

In [ ]:
handler = agent_workflow.run(
    user_msg="I want to retrieve some data from a URL. Suggest a method to do so."
)

In [ ]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

current_agent = None
current_tool_calls = ""

async for event in handler.stream_events():
    if (
        hasattr(event, "current_agent_name")
        and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'='*50}\n")

    # if isinstance(event, AgentStream):
    #     if event.delta:
    #         print(event.delta, end="", flush=True)
    # elif isinstance(event, AgentInput):
    #     print("📥 Input:", event.input)
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")


🤖 Agent: Agent

📤 Output: Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: retrieve_relevant_root_modules
Action Input: {"query": "retrieve data from URL"}

🛠️  Planning to use tools: ['retrieve_relevant_root_modules']
🔨 Calling Tool: retrieve_relevant_root_modules
  With arguments: {'query': 'retrieve data from URL'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'retrieve data from URL'}
  Output: Relevant Root Modules:

1. **urllib**: ### Summary of `urllib` Module

#### Overview
The `urllib` module in Python is a package that collects several modules for working with URLs. It provides tools for opening URLs using various protocols and includes functionalities for parsing and manipulating URLs.

#### Sub-Modules

1. **urllib.request**
   - Provides classes and functions for opening and reading URLs. It handles HTTP, FTP, and other protocols.
   - Includes classes like `Request`, `OpenerDirector`, and `HTTPHandler` for customizing and managing requests.
   - Supports automatic redirection, handling of HTTP errors, and proxy support.

2. **urllib.response**
   - Provides classes for handling HTTP responses in a file-like manner.
   - Includes base classes like `addbase`, `addclosehook`, `addinfo`, and `addinfourl` for different functionalities.
   - Key classes like `addinfo

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Root Modules:

1. **requests**: ### Summary of `requests`

#### Overview
`requests` is a Python HTTP library designed to simplify the process of making HTTP requests. It provides a user-friendly interface for interacting with web services and is widely used due to its ease of use and robust feature set.

#### Key Functionalities
1. **HTTP Requests**: Supports various HTTP methods such as GET, POST, PUT, DELETE, etc.
2. **Session Management**: Manages cookies, headers, and other settings across multiple requests using sessions.
3. **Authentication**: Supports basic, proxy, and digest authentication.
4. **Proxy Support**: Handles HTTP and SOCKS proxies.
5. **SSL Verification**: Performs secure connections with automatic verification.
6. **Content Handling**: Automatically decompresses and decodes content.
7. **File Uploads**: Facilitates multi-part file uploads.
8. **Timeouts**: Allows specifying connection and read timeouts.
9. **Streaming**: Supports do

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

INFO:root:Dependencies found: ['']


Dependencies found: ['']


INFO:root:Next level dependencies: []


Next level dependencies: []
🔧 Tool Result (generate_code):
  Arguments: {'name': 'requests.get', 'label': 'Retrieve data from a URL', 'other_instructions': 'Provide an example usage to retrieve data from a URL and print the content.'}
  Output: ```python
import requests

def main():
    """
    This script demonstrates the usage of requests.get to retrieve data from a URL
    and print the content.
    """
    try:
        # Replace 'https://www.example.com' with the URL you want to access
        response = requests.get('https://www.example.com')

        # Raise HTTPError for bad responses (4xx or 5xx)
        response.raise_for_status()

        # Print the content of the response
        print(response.text)

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()
```
📤 Output: Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: To retrieve data from a URL in